<a href="https://colab.research.google.com/github/cksleigen/lg-aimers-demand-forecasting/blob/chanhee/TCN_%EB%8B%A8%EC%9D%BC_%ED%85%8C%EC%8A%A4%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 그냥 TCN
[TCN] [Epoch 050] train_loss: 0.25508  val_loss: 0.38320


In [4]:
# tcn_only.py
# -*- coding: utf-8 -*-
import os, gc, math, random, warnings, json
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# =====================
# Config (경로/하이퍼파라미터)
# =====================
@dataclass
class EnhancedNHiTSConfig:
    # Paths
    train_csv: str = "/content/drive/MyDrive/data/train/train_original.csv"
    test_glob: str = "/content/drive/MyDrive/data/test/*.csv"
    submission_template_csv: str = "/content/drive/MyDrive/data/sample_submission.csv"
    out_submission_csv: str = "/content/drive/MyDrive/data/0823_submission.csv"
    checkpoint_dir: str = "/content/drive/MyDrive/data/checkpoint/enhanced_checkpoints"
    # Columns
    date_col: str = "영업일자"
    item_col: str = "영업장명_메뉴명"
    target_col: str = "매출수량"
    # Window
    in_len: int = 28
    out_len: int = 7
    # Train filtering
    train_end_date: str = "2024-06-15"
    # General
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log1p: bool = True
    # Dataloaders
    num_workers: int = 4
    pin_memory: bool = True
    persistent_workers: bool = False
    # Loss
    eps_smape: float = 0.01
    zero_weight: float = 0.01
    hurdle_lambda: float = 0.15
    # AMP/EMA
    use_amp: bool = True
    use_compile: bool = False
    ema_decay: float = 0.9998
    # Train hyperparams
    EPOCHS_FULL: int = 50
    BATCH_FULL: int = 256
    BASE_LR_FULL: float = 6e-4
    MAX_LR_FULL: float = 1.5e-3
    WD_FULL: float = 6e-4
    # Early stop / grad clip
    grad_clip: float = 0.5
    earlystop_patience_ratio: float = 0.12
    earlystop_patience_min: int = 8
    # Optional weights/holidays
    store_weights: Optional[Dict[str, float]] = None
    custom_holidays_list: Optional[List[str]] = None

DEFAULT_STORE_WEIGHTS = {
    "미라시아": 1, "담하": 1, "연회장": 1, "라그로타": 1,
    "느티나무 셀프BBQ": 1, "화담숲주막": 1, "카페테리아": 1,
    "화담숲카페": 1, "포레스트릿": 1,
}
DEFAULT_CUSTOM_HOLIDAYS = [
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-01',
    '2023-05-05','2023-05-27','2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30',
    '2023-10-02','2023-10-03','2023-10-09','2023-12-25',
    '2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01','2024-04-10',
    '2024-05-01','2024-05-05','2024-05-06','2024-05-15','2024-06-06','2024-08-15','2024-09-16',
    '2024-09-17','2024-09-18','2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    '2025-01-01','2025-01-28','2025-01-29','2025-01-30','2025-03-01','2025-03-03','2025-05-01',
    '2025-05-05','2025-05-06','2025-06-06','2025-08-15'
]

# =====================
# Utils
# =====================
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sine_cosine_encoding(value: float, max_val: float):
    return math.sin(2 * math.pi * value / max_val), math.cos(2 * math.pi * value / max_val)

def parse_store_name(item_full: str) -> str:
    return item_full.split("_")[0]

def parse_menu_name(item_full: str) -> str:
    return item_full.split("_", 1)[1] if "_" in item_full else item_full

def build_enhanced_features(dates: List[pd.Timestamp], holidays_set: set,
                            store_names: Optional[List[str]] = None) -> pd.DataFrame:
    df = pd.DataFrame({"date": dates})
    df["dow"] = df["date"].dt.weekday
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["week"] = df["date"].apply(lambda d: int(d.isocalendar().week))
    df["quarter"] = df["date"].dt.quarter
    df["is_friday"] = (df["dow"] == 4).astype(int)
    df["is_saturday"] = (df["dow"] == 5).astype(int)
    df["is_sunday"] = (df["dow"] == 6).astype(int)
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["is_holiday"] = df["date"].isin(holidays_set).astype(int)
    df["is_off"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)
    df["tomorrow"] = df["date"] + pd.Timedelta(days=1)
    df["yesterday"] = df["date"] - pd.Timedelta(days=1)
    df["is_before_holiday"] = df["tomorrow"].isin(holidays_set).astype(int)
    df["is_after_holiday"] = df["yesterday"].isin(holidays_set).astype(int)
    df["is_month_start"] = (df["day"] <= 3).astype(int)
    df["is_month_end"] = (df["day"] >= 28).astype(int)
    df["is_spring"] = df["month"].isin([4, 5, 6]).astype(int)
    df["is_summer"] = df["month"].isin([7, 8]).astype(int)
    df["is_autumn"] = df["month"].isin([9, 10, 11]).astype(int)
    df["is_winter"] = df["month"].isin([12, 1, 2, 3]).astype(int)
    df["is_summer_vacation"] = df["month"].isin([7, 8]).astype(int)
    df["is_winter_vacation"] = df["month"].isin([12, 1, 2]).astype(int)

    # 날짜 기반 정기 휴무일 마스크 (원본 로직 요약)
    df['is_regular_holiday'] = 0
    df.loc[((df['date'].dt.month.isin(range(2, 12))) & (df['date'].dt.year == 2023) & (df['dow'] == 0)) |
           ((df['date'].dt.month.isin(range(3, 7))) & (df['date'].dt.year == 2024) & (df['dow'] == 0)) |
           (df['date'].isin(['2023-03-01'])) |
           (df['date'].isin(['2023-09-01','2023-09-02','2023-09-03']) & (df['dow'].isin([0,1,2]))) |
           (df['date'].isin(['2024-03-01'])) |
           ((df['date'] >= '2023-05-01') & (df['dow'].isin([0,1,2,3]))) |
           (df['date'].dt.month.isin([12,1,2,3]) & df['date'].dt.year.isin([2023,2024,2025])) |
           ((~df['date'].dt.month.isin([12,1,2,3])) & (df['dow'] == 0)), 'is_regular_holiday'] = 1

    df[["month_sin", "month_cos"]] = df["month"].apply(lambda m: pd.Series(sine_cosine_encoding(m, 12)))
    df[["dow_sin", "dow_cos"]] = df["dow"].apply(lambda d: pd.Series(sine_cosine_encoding(d, 7)))
    df[["doy_sin", "doy_cos"]] = df["date"].apply(lambda d: pd.Series(sine_cosine_encoding(d.dayofyear, 365)))
    df[["week_sin", "week_cos"]] = df["week"].apply(lambda w: pd.Series(sine_cosine_encoding(w, 52)))
    df[["quarter_sin", "quarter_cos"]] = df["quarter"].apply(lambda q: pd.Series(sine_cosine_encoding(q, 4)))
    return df.drop(columns=["tomorrow", "yesterday"])

def get_menu_category(menu_name: str) -> str:
    menu_lower = menu_name.lower()
    if any(word in menu_lower for word in ['막걸리','소주','맥주','와인','참이슬','처음처럼','카스','하이네켄','버드와이저','스텔라']):
        return 'alcohol'
    elif any(word in menu_lower for word in ['찌개','탕','국밥','라면','해장국','갈비탕']):
        return 'hot_food'
    elif any(word in menu_lower for word in ['삼겹','갈비','목살','bbq','구이','불고기']):
        return 'bbq'
    elif any(word in menu_lower for word in ['아이스크림','식혜','콜라','스프라이트','에이드']):
        return 'dessert_drink'
    elif any(word in menu_lower for word in ['아메리카노','라떼','커피']):
        return 'coffee'
    elif any(word in menu_lower for word in ['냉면','파스타','스파게티','면','우동']):
        return 'noodles'
    elif any(word in menu_lower for word in ['비빔밥','볶음밥','공깃밥','정식']):
        return 'rice'
    else:
        return 'others'

def get_store_type(store_name: str) -> str:
    if store_name in ["느티나무 셀프BBQ", "연회장"]:
        return 'Special_Occasion'
    elif store_name in ["화담숲주막", "화담숲카페"]:
        return 'Forest'
    elif store_name in ["담하", "미라시아"]:
        return 'Fine_Dining'
    elif store_name in ["카페테리아", "포레스트릿"]:
        return 'Casual'
    else:
        return 'Unique_Venue'

def get_weekend_sales_ratio(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy['영업일자'] = pd.to_datetime(df_copy['영업일자'])
    df_copy['요일'] = df_copy['영업일자'].dt.weekday
    weekend_df = df_copy[df_copy['요일'].isin([5,6])]
    weekday_df = df_copy[~df_copy['요일'].isin([5,6])]
    weekend_sales = weekend_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    weekday_sales = weekday_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    sales_ratio = (weekend_sales / weekday_sales).fillna(1.0).reset_index()
    sales_ratio.rename(columns={'매출수량':'weekend_sales_ratio'}, inplace=True)
    sales_ratio['weekend_sales_ratio'] = sales_ratio['weekend_sales_ratio'].replace([float('inf'), -float('inf')], 1.0)
    return sales_ratio

def make_val_mask_by_week(dataset: 'EnhancedNHiTSDataset', end_date_str: str) -> np.ndarray:
    end_date = pd.to_datetime(end_date_str)
    start_date = end_date - pd.Timedelta(days=6)
    ted = dataset.target_end_dates
    return (ted >= start_date) & (ted <= end_date)

# =====================
# Dataset
# =====================
class EnhancedNHiTSDataset(Dataset):
    """
    원본의 Dense 보강/캘린더피처/주말비율/인덱싱 로직을 그대로 사용
    """
    def __init__(self, cfg: EnhancedNHiTSConfig, df: pd.DataFrame):
        self.cfg = cfg
        self.df = df.copy()
        if cfg.date_col not in self.df.columns or cfg.item_col not in self.df.columns or cfg.target_col not in self.df.columns:
            raise ValueError("입력 데이터에 필요한 컬럼이 없습니다.")
        self.df[cfg.date_col] = pd.to_datetime(self.df[cfg.date_col])
        self.df[cfg.target_col] = pd.to_numeric(self.df[cfg.target_col], errors='coerce').fillna(0.0).clip(lower=0)

        # Dense dataset 보강 (모든 날짜×아이템)
        all_dates = pd.date_range(start=self.df[cfg.date_col].min(), end=self.df[cfg.date_col].max())
        all_items = self.df[cfg.item_col].unique()
        full_df = pd.MultiIndex.from_product([all_dates, all_items], names=[cfg.date_col, cfg.item_col]).to_frame(index=False)
        self.df = pd.merge(full_df, self.df, on=[cfg.date_col, cfg.item_col], how='left')
        self.df[cfg.target_col] = self.df[cfg.target_col].fillna(0)

        # 주말 상대 판매량 비율 병합
        weekend_ratio_df = get_weekend_sales_ratio(self.df)
        self.df = pd.merge(self.df, weekend_ratio_df, on=cfg.item_col, how='left')
        self.df['weekend_sales_ratio'].fillna(1.0, inplace=True)

        pivot = self.df.pivot(index=cfg.date_col, columns=cfg.item_col, values=cfg.target_col).sort_index()
        self.items = list(pivot.columns)
        self.dates = list(pivot.index)
        self.values = pivot.fillna(0.0).values.astype(np.float32)

        holidays_set = set(pd.to_datetime(cfg.custom_holidays_list)) if cfg.custom_holidays_list else set()
        caldf = build_enhanced_features(self.dates, holidays_set)
        self.cal_feats = caldf.drop(columns=["date"]).values.astype(np.float32)
        self.cal_feat_names = [c for c in caldf.columns if c != "date"]

        stores = [parse_store_name(it) for it in self.items]
        menus  = [parse_menu_name(it) for it in self.items]

        self.store2idx = {s: i for i, s in enumerate(sorted(set(stores)))}
        self.item_store_idx = np.array([self.store2idx[s] for s in stores], dtype=np.int64)
        self.n_stores = len(self.store2idx)

        menu_categories = [get_menu_category(m) for m in menus]
        self.cat2idx = {c: i for i, c in enumerate(sorted(set(menu_categories)))}
        self.item_cat_idx = np.array([self.cat2idx[c] for c in menu_categories], dtype=np.int64)
        self.n_categories = len(self.cat2idx)

        store_types = [get_store_type(s) for s in stores]
        self.type2idx = {t: i for i, t in enumerate(sorted(set(store_types)))}
        self.item_type_idx = np.array([self.type2idx[t] for t in store_types], dtype=np.int64)
        self.n_types = len(self.type2idx)

        # 주말 비율 배열화
        self.item_weekend_ratio = (
            self.df.drop_duplicates(subset=[cfg.item_col])
                  .set_index(cfg.item_col)
                  .loc[self.items, 'weekend_sales_ratio'].values.astype(np.float32)
        )

        sw = cfg.store_weights or {}
        self.sample_weights = np.array([sw.get(parse_store_name(it), 1.0) for it in self.items], dtype=np.float32)

        self.indices: List[Tuple[int,int]] = []
        self.target_end_dates: List[pd.Timestamp] = []
        T = len(self.dates)
        Lx, Ly = cfg.in_len, cfg.out_len
        cutoff = pd.to_datetime(cfg.train_end_date)
        max_start = max(0, T - (Lx + Ly))

        for j in range(len(self.items)):
            for t0 in range(0, max_start + 1):
                end_date = self.dates[t0 + Lx + Ly - 1]
                if end_date <= cutoff:
                    self.indices.append((t0, j))
                    self.target_end_dates.append(end_date)

        self.target_end_dates = np.asarray(self.target_end_dates, dtype='datetime64[ns]')

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        cfg = self.cfg
        t0, j = self.indices[idx]
        Lx, Ly = cfg.in_len, cfg.out_len

        x = self.values[t0:t0 + Lx, j]
        y = self.values[t0 + Lx:t0 + Lx + Ly, j]

        if cfg.log1p:
            x_in = np.log1p(x); y_out = np.log1p(y)
        else:
            x_in, y_out = x.copy(), y.copy()

        past_cal = self.cal_feats[t0:t0 + Lx, :]
        fut_cal  = self.cal_feats[t0 + Lx:t0 + Lx + Ly, :]

        store_idx = self.item_store_idx[j]
        cat_idx   = self.item_cat_idx[j]
        type_idx  = self.item_type_idx[j]
        sample_w  = self.sample_weights[j]
        weekend_ratio = self.item_weekend_ratio[j]

        pos_mask = (y > 0).astype(np.float32)

        zero_ratio = np.mean(x == 0)
        weekend_factor = np.mean(past_cal[:, 12])

        return {
            "x": torch.from_numpy(x_in).float(),
            "y": torch.from_numpy(y_out).float(),
            "past_cal": torch.from_numpy(past_cal).float(),
            "fut_cal": torch.from_numpy(fut_cal).float(),
            "store_idx": torch.tensor(store_idx, dtype=torch.long),
            "cat_idx": torch.tensor(cat_idx, dtype=torch.long),
            "type_idx": torch.tensor(type_idx, dtype=torch.long),
            "sample_w": torch.tensor(sample_w, dtype=torch.float32),
            "pos_mask": torch.from_numpy(pos_mask).float(),
            "zero_ratio": torch.tensor(zero_ratio, dtype=torch.float32),
            "weekend_factor": torch.tensor(weekend_factor, dtype=torch.float32),
            "weekend_ratio": torch.tensor(weekend_ratio, dtype=torch.float32),
        }
# (↑ 데이터셋/피처 로직은 원본과 동일합니다. :contentReference[oaicite:0]{index=0})

# =====================
# TCN (시퀀스 헤드 + 메타 헤드 + 확률 헤드)
# =====================
class _TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3, d=1, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(c_out, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(),
        )
        self.proj = nn.Conv1d(c_in, c_out, kernel_size=1) if c_in!=c_out else nn.Identity()
        self.norm = nn.LayerNorm(c_out)

    def forward(self, x):
        y = self.net(x) + self.proj(x)
        return self.norm(y.transpose(1,2)).transpose(1,2)
# (↑ 블록 구현은 원본과 동일. :contentReference[oaicite:1]{index=1})

class TCNTiny(nn.Module):
    def __init__(self, cfg, in_len, out_len, cal_dim, n_stores, n_categories, n_types, C=128, depth=4, drop=0.1):
        super().__init__()
        self.out_len = out_len
        self.stem = nn.Conv1d(1, C, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([_TCNBlock(C, C, k=3, d=2**i, p=drop) for i in range(depth)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(C, out_len))

        self.store_emb = nn.Embedding(n_stores,64)
        self.cat_emb   = nn.Embedding(n_categories,32)
        self.type_emb  = nn.Embedding(n_types,16)
        self.cal_proj  = nn.Linear(cal_dim,128)

        self.meta_head = nn.Sequential(nn.Linear(64+32+16+128, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))
        self.prob_head = nn.Sequential(nn.Linear(64+32+16+128+1, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))

    def forward(self, x, past_cal, fut_cal, store_idx, cat_idx, type_idx, **kwargs):
        z = self.stem(x.unsqueeze(1))
        for blk in self.blocks:
            z = blk(z)
        seq_out = self.head(z)

        store = self.store_emb(store_idx); cat = self.cat_emb(cat_idx); typ = self.type_emb(type_idx)
        cal_all = torch.cat([past_cal, fut_cal], dim=1).mean(dim=1); cal = self.cal_proj(cal_all)
        meta = torch.cat([store,cat,typ,cal], dim=-1)

        value_pred  = seq_out + self.meta_head(meta)
        prob_logits = self.prob_head(torch.cat([meta, x.mean(dim=1, keepdim=True)], dim=-1))
        return value_pred, prob_logits
# (↑ TCN 헤드/메타/확률 구조는 원본과 동일. :contentReference[oaicite:2]{index=2})

# =====================
# Loss / EMA
# =====================
class UltraEnhancedHurdleLoss(nn.Module):
    """
    v_pred_log, y_true_log는 log1p 스케일을 가정.
    """
    def __init__(self, eps: float = 0.01, zero_weight: float = 0.01, lambda_bce: float = 0.15):
        super().__init__()
        self.eps = eps; self.zero_weight = zero_weight; self.lambda_bce = lambda_bce
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, v_pred_log, p_logits, y_true_log, pos_mask, sample_w):
        z = (pos_mask > 0).float()
        bce = self.bce(p_logits, z)

        yp_val = torch.expm1(v_pred_log).clamp_min(0.0)
        yt_val = torch.expm1(y_true_log).clamp_min(0.0)

        ultra_eps = torch.where(yt_val < 0.5, self.eps * 0.2,
                                torch.where(yt_val < 2.0, self.eps * 0.5, self.eps))
        denom = (torch.abs(yp_val) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_pos = 2.0 * torch.abs(yp_val - yt_val) / denom
        smape_pos = smape_pos * (yt_val > 0).float()

        p = torch.sigmoid(p_logits)
        y_hat = p * yp_val
        denom2 = (torch.abs(y_hat) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_all = 2.0 * torch.abs(y_hat - yt_val) / denom2

        ultra_zero_weight = torch.where(
            yt_val < 0.01, torch.full_like(yt_val, self.zero_weight * 0.1),
            torch.where(yt_val < 0.1, torch.full_like(yt_val, self.zero_weight * 0.3),
                        torch.where(yt_val < 1.0, torch.full_like(yt_val, self.zero_weight * 0.6), torch.ones_like(yt_val)))
        )
        smape_all = smape_all * ultra_zero_weight

        bce_s = bce.mean(dim=1); pos_s = smape_pos.mean(dim=1); all_s = smape_all.mean(dim=1)
        sample_loss = self.lambda_bce * bce_s + 0.4 * pos_s + 0.6 * all_s

        sw = sample_w.view(-1); wsum = sw.sum().clamp_min(1e-8)
        loss = (sample_loss * sw).sum() / wsum
        return loss, sample_loss.detach(), sw.detach()
# (↑ 허들 로스는 원본과 동일 계산. :contentReference[oaicite:3]{index=3})

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.shadow = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

    def update(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=1-self.decay)

    def apply_to(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    p.copy_(self.shadow[name])
# (↑ EMA는 원본과 동일. :contentReference[oaicite:4]{index=4})

# =====================
# Trainer
# =====================
class GenericTrainer:
    def __init__(self, cfg: EnhancedNHiTSConfig, dataset: EnhancedNHiTSDataset, model: nn.Module,
                 epochs: int, batch_size: int, base_lr: float, max_lr: float, weight_decay: float):
        self.cfg = cfg; self.dataset = dataset; self.model = model.to(cfg.device)
        self.device = torch.device(cfg.device)
        self.criterion = UltraEnhancedHurdleLoss(cfg.eps_smape, cfg.zero_weight, cfg.hurdle_lambda)
        self.ema = EMA(self.model, decay=cfg.ema_decay)
        self.epochs = epochs; self.batch_size = batch_size
        self.base_lr = base_lr; self.max_lr = max_lr; self.weight_decay = weight_decay

    def make_loaders_from_mask(self, mask_val: np.ndarray):
        idx_all = np.arange(len(self.dataset))
        val_idx = idx_all[mask_val]; train_idx = idx_all[~mask_val]
        train_subset = torch.utils.data.Subset(self.dataset, train_idx)
        val_subset = torch.utils.data.Subset(self.dataset, val_idx)
        train_loader = DataLoader(
            train_subset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        val_loader = DataLoader(
            val_subset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        return train_loader, val_loader
    # (↑ 데이터로더 구성은 원본과 동일. :contentReference[oaicite:5]{index=5})

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, use_ema: bool = True) -> float:
        if loader is None: return float('inf')
        self.model.eval()
        backup = None
        if use_ema:
            backup = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            self.ema.apply_to(self.model)

        total_loss, total_weight = 0.0, 0.0
        amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
            self.cfg.use_amp and torch.cuda.is_available() and self.device.type == "cuda"
        ) else torch.cuda.amp.autocast(enabled=False)

        with amp_ctx:
            for batch in loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)
                weekend_ratio = batch["weekend_ratio"].to(self.device)

                v_pred, p_logits = self.model(x, past_cal, fut_cal, store_idx, cat_idx, type_idx, weekend_ratio=weekend_ratio)
                _, sample_loss, sw = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                total_loss += (sample_loss * sw).sum().item()
                total_weight += sw.sum().item()

        if use_ema and backup is not None:
            self.model.load_state_dict(backup); del backup
            gc.collect(); torch.cuda.empty_cache()

        if total_weight <= 0: return float('inf')
        return total_loss / total_weight
    # (↑ 평가 로직은 원본과 동일한 형태. :contentReference[oaicite:6]{index=6})

    def train_with_loaders(self, train_loader: DataLoader, val_loader: DataLoader, model_name: str = "TCN"):
        optim = torch.optim.AdamW(self.model.parameters(), lr=self.base_lr, weight_decay=self.weight_decay, betas=(0.9, 0.999))
        steps_per_epoch = max(1, len(train_loader))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            optim, max_lr=self.max_lr, epochs=self.epochs,
            steps_per_epoch=steps_per_epoch, pct_start=0.05,
            div_factor=max(1e-8, self.max_lr / max(1e-8, self.base_lr))
        )
        patience = max(self.cfg.earlystop_patience_min, int(self.epochs * self.cfg.earlystop_patience_ratio))
        best_val = float("inf"); best_state = None; no_improve = 0

        for epoch in range(1, self.epochs+1):
            self.model.train()
            amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
                self.cfg.use_amp and torch.cuda.is_available() and self.device.type=="cuda"
            ) else torch.cuda.amp.autocast(enabled=False)
            running = []
            for batch in train_loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)
                weekend_ratio = batch["weekend_ratio"].to(self.device)

                with amp_ctx:
                    v_pred, p_logits = self.model(x, past_cal, fut_cal, store_idx, cat_idx, type_idx, weekend_ratio=weekend_ratio)
                    loss, _, _ = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                optim.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                optim.step(); sched.step(); self.ema.update(self.model)
                running.append(loss.detach().item())

            val_loss = self.evaluate(val_loader, use_ema=True)
            tr_mean = float(np.mean(running)) if running else float('nan')
            print(f"[{model_name}] [Epoch {epoch:03d}] train_loss: {tr_mean:.5f}  val_loss: {val_loss:.5f}")

            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch} (patience={patience})")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state, strict=True)
        return self.model, best_val
# (↑ 학습 루프/OneCycleLR/EMA 업데이트/조기종료는 원본 트레이너와 동일. :contentReference[oaicite:7]{index=7})

# =====================
# Build helper (TCN만)
# =====================
def build_tcn(cfg: EnhancedNHiTSConfig, ds: EnhancedNHiTSDataset, C=128, depth=4, drop=0.1):
    return TCNTiny(cfg, cfg.in_len, cfg.out_len, ds.cal_feats.shape[1], ds.n_stores, ds.n_categories, ds.n_types, C=C, depth=depth, drop=drop)
# (↑ 원본 build_model_by_name의 TCN 분기와 동일 파라미터. :contentReference[oaicite:8]{index=8})

# =====================
# Main
# =====================
if __name__ == "__main__":
    cfg = EnhancedNHiTSConfig()
    if cfg.store_weights is None: cfg.store_weights = DEFAULT_STORE_WEIGHTS
    if cfg.custom_holidays_list is None: cfg.custom_holidays_list = DEFAULT_CUSTOM_HOLIDAYS
    set_seed(cfg.seed)

    if not os.path.exists(cfg.train_csv):
        raise FileNotFoundError(f"학습 파일을 찾을 수 없습니다: {cfg.train_csv}")

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    # Load train
    train_df = pd.read_csv(cfg.train_csv)
    ds = EnhancedNHiTSDataset(cfg, train_df)

    # CV mask (원본은 여러 fold; 여기선 첫 fold 예시)
    cv_end = "2024-06-14"
    mask_val = make_val_mask_by_week(ds, cv_end)

    # Build TCN
    model = build_tcn(cfg, ds, C=128, depth=4, drop=0.1)

    # Train
    trainer = GenericTrainer(cfg, ds, model,
                             epochs=cfg.EPOCHS_FULL, batch_size=cfg.BATCH_FULL,
                             base_lr=cfg.BASE_LR_FULL, max_lr=cfg.MAX_LR_FULL, weight_decay=cfg.WD_FULL)
    train_loader, val_loader = trainer.make_loaders_from_mask(mask_val)
    model, best_val = trainer.train_with_loaders(train_loader, val_loader, model_name="TCN")

    # Save checkpoint
    ckpt_path = os.path.join(cfg.checkpoint_dir, "TCN_best_fold.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Saved best checkpoint to {ckpt_path} (val_loss={best_val:.6f})")


[TCN] [Epoch 001] train_loss: 0.41247  val_loss: 0.86912
[TCN] [Epoch 002] train_loss: 0.35934  val_loss: 0.80881
[TCN] [Epoch 003] train_loss: 0.34688  val_loss: 0.76818
[TCN] [Epoch 004] train_loss: 0.33888  val_loss: 0.74660
[TCN] [Epoch 005] train_loss: 0.33247  val_loss: 0.74323
[TCN] [Epoch 006] train_loss: 0.32781  val_loss: 0.74507
[TCN] [Epoch 007] train_loss: 0.32353  val_loss: 0.74223
[TCN] [Epoch 008] train_loss: 0.31932  val_loss: 0.73189
[TCN] [Epoch 009] train_loss: 0.31562  val_loss: 0.71493
[TCN] [Epoch 010] train_loss: 0.31182  val_loss: 0.69115
[TCN] [Epoch 011] train_loss: 0.30864  val_loss: 0.66332
[TCN] [Epoch 012] train_loss: 0.30525  val_loss: 0.63376
[TCN] [Epoch 013] train_loss: 0.30197  val_loss: 0.60440
[TCN] [Epoch 014] train_loss: 0.29919  val_loss: 0.57751
[TCN] [Epoch 015] train_loss: 0.29655  val_loss: 0.55387
[TCN] [Epoch 016] train_loss: 0.29381  val_loss: 0.53288
[TCN] [Epoch 017] train_loss: 0.29146  val_loss: 0.51477
[TCN] [Epoch 018] train_loss: 0

# 상관관계 피처 도입
[TCN] [Epoch 050] train_loss: 0.25846  val_loss: 0.38108


In [3]:
# tcn_only_with_corr_and_dow.py
# -*- coding: utf-8 -*-
import os, gc, math, random, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# =====================
# Config
# =====================
@dataclass
class EnhancedNHiTSConfig:
    # Paths
    train_csv: str = "/content/drive/MyDrive/data/train/train_original.csv"
    test_glob: str = "/content/drive/MyDrive/data/test/*.csv"
    submission_template_csv: str = "/content/drive/MyDrive/data/sample_submission.csv"
    out_submission_csv: str = "/content/drive/MyDrive/data/0823_submission.csv"
    checkpoint_dir: str = "/content/drive/MyDrive/data/checkpoint/enhanced_checkpoints"

    # Columns
    date_col: str = "영업일자"
    item_col: str = "영업장명_메뉴명"
    target_col: str = "매출수량"

    # Window
    in_len: int = 28
    out_len: int = 7

    # Train filtering
    train_end_date: str = "2024-06-15"

    # General
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log1p: bool = True

    # Dataloaders
    num_workers: int = 4
    pin_memory: bool = True
    persistent_workers: bool = False

    # Loss
    eps_smape: float = 0.01
    zero_weight: float = 0.01
    hurdle_lambda: float = 0.15

    # AMP/EMA
    use_amp: bool = True
    use_compile: bool = False
    ema_decay: float = 0.9998

    # Train hyperparams
    EPOCHS_FULL: int = 50
    BATCH_FULL: int = 256
    BASE_LR_FULL: float = 6e-4
    MAX_LR_FULL: float = 1.5e-3
    WD_FULL: float = 6e-4

    # Early stop / grad clip
    grad_clip: float = 0.5
    earlystop_patience_ratio: float = 0.12
    earlystop_patience_min: int = 8

    # Optional weights/holidays (채워지지 않으면 기본 사용)
    store_weights: Optional[Dict[str, float]] = None
    custom_holidays_list: Optional[List[str]] = None

    # ====== NEW: Feature toggles & params ======
    # Corr-based features
    USE_CORR_FEATURES: bool = True
    CORR_THRESHOLD: float = 0.5     # 상관 임계치
    CORR_TOPN: int = 5              # 메뉴당 상위 파트너 N개 제한 (0 = 제한 없음)
    CORR_LAGS: Tuple[int, ...] = (1, 7)
    CORR_RMEANS: Tuple[int, ...] = (7, 14)  # 모두 shift=1

    # Day-of-week strength
    USE_DOW_STRENGTH: bool = False
    DOW_RATIO_THRESHOLD: float = 2.0    # 평균 대비 2배 이상
    DOW_MIN_SUPPORT: int = 4            # 요일 평균 계산에 필요한 최소 관측일 수

    # Scaling extra features: "none" | "log1p" | "zscore"
    EXTRA_FEAT_SCALING: str = "none"


DEFAULT_STORE_WEIGHTS = {
    "미라시아": 1, "담하": 1, "연회장": 1, "라그로타": 1,
    "느티나무 셀프BBQ": 1, "화담숲주막": 1, "카페테리아": 1,
    "화담숲카페": 1, "포레스트릿": 1,
}
DEFAULT_CUSTOM_HOLIDAYS = [
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-01',
    '2023-05-05','2023-05-27','2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30',
    '2023-10-02','2023-10-03','2023-10-09','2023-12-25',
    '2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01','2024-04-10',
    '2024-05-01','2024-05-05','2024-05-06','2024-05-15','2024-06-06','2024-08-15','2024-09-16',
    '2024-09-17','2024-09-18','2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    '2025-01-01','2025-01-28','2025-01-29','2025-01-30','2025-03-01','2025-03-03','2025-05-01',
    '2025-05-05','2025-05-06','2025-06-06','2025-08-15'
]

# =====================
# Utils
# =====================
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sine_cosine_encoding(value: float, max_val: float):
    return math.sin(2 * math.pi * value / max_val), math.cos(2 * math.pi * value / max_val)

def parse_store_name(item_full: str) -> str:
    return item_full.split("_")[0]

def parse_menu_name(item_full: str) -> str:
    return item_full.split("_", 1)[1] if "_" in item_full else item_full

def build_enhanced_features(dates: List[pd.Timestamp], holidays_set: set,
                            store_names: Optional[List[str]] = None) -> pd.DataFrame:
    df = pd.DataFrame({"date": dates})
    df["dow"] = df["date"].dt.weekday
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["week"] = df["date"].apply(lambda d: int(d.isocalendar().week))
    df["quarter"] = df["date"].dt.quarter
    df["is_friday"] = (df["dow"] == 4).astype(int)
    df["is_saturday"] = (df["dow"] == 5).astype(int)
    df["is_sunday"] = (df["dow"] == 6).astype(int)
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["is_holiday"] = df["date"].isin(holidays_set).astype(int)
    df["is_off"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)
    df["tomorrow"] = df["date"] + pd.Timedelta(days=1)
    df["yesterday"] = df["date"] - pd.Timedelta(days=1)
    df["is_before_holiday"] = df["tomorrow"].isin(holidays_set).astype(int)
    df["is_after_holiday"] = df["yesterday"].isin(holidays_set).astype(int)
    df["is_month_start"] = (df["day"] <= 3).astype(int)
    df["is_month_end"] = (df["day"] >= 28).astype(int)
    df["is_spring"] = df["month"].isin([4, 5, 6]).astype(int)
    df["is_summer"] = df["month"].isin([7, 8]).astype(int)
    df["is_autumn"] = df["month"].isin([9, 10, 11]).astype(int)
    df["is_winter"] = df["month"].isin([12, 1, 2, 3]).astype(int)
    df["is_summer_vacation"] = df["month"].isin([7, 8]).astype(int)
    df["is_winter_vacation"] = df["month"].isin([12, 1, 2]).astype(int)

    # 날짜 기반 정기 휴무일 (원본 로직 요약)
    df['is_regular_holiday'] = 0
    df.loc[((df['date'].dt.month.isin(range(2, 12))) & (df['date'].dt.year == 2023) & (df['dow'] == 0)) |
           ((df['date'].dt.month.isin(range(3, 7))) & (df['date'].dt.year == 2024) & (df['dow'] == 0)) |
           (df['date'].isin(['2023-03-01'])) |
           (df['date'].isin(['2023-09-01','2023-09-02','2023-09-03']) & (df['dow'].isin([0,1,2]))) |
           (df['date'].isin(['2024-03-01'])) |
           ((df['date'] >= '2023-05-01') & (df['dow'].isin([0,1,2,3]))) |
           (df['date'].dt.month.isin([12,1,2,3]) & df['date'].dt.year.isin([2023,2024,2025])) |
           ((~df['date'].dt.month.isin([12,1,2,3])) & (df['dow'] == 0)), 'is_regular_holiday'] = 1

    df[["month_sin", "month_cos"]] = df["month"].apply(lambda m: pd.Series(sine_cosine_encoding(m, 12)))
    df[["dow_sin", "dow_cos"]] = df["dow"].apply(lambda d: pd.Series(sine_cosine_encoding(d, 7)))
    df[["doy_sin", "doy_cos"]] = df["date"].apply(lambda d: pd.Series(sine_cosine_encoding(d.dayofyear, 365)))
    df[["week_sin", "week_cos"]] = df["week"].apply(lambda w: pd.Series(sine_cosine_encoding(w, 52)))
    df[["quarter_sin", "quarter_cos"]] = df["quarter"].apply(lambda q: pd.Series(sine_cosine_encoding(q, 4)))
    return df.drop(columns=["tomorrow", "yesterday"])

def get_menu_category(menu_name: str) -> str:
    menu_lower = menu_name.lower()
    if any(word in menu_lower for word in ['막걸리','소주','맥주','와인','참이슬','처음처럼','카스','하이네켄','버드와이저','스텔라']):
        return 'alcohol'
    elif any(word in menu_lower for word in ['찌개','탕','국밥','라면','해장국','갈비탕']):
        return 'hot_food'
    elif any(word in menu_lower for word in ['삼겹','갈비','목살','bbq','구이','불고기']):
        return 'bbq'
    elif any(word in menu_lower for word in ['아이스크림','식혜','콜라','스프라이트','에이드']):
        return 'dessert_drink'
    elif any(word in menu_lower for word in ['아메리카노','라떼','커피']):
        return 'coffee'
    elif any(word in menu_lower for word in ['냉면','파스타','스파게티','면','우동']):
        return 'noodles'
    elif any(word in menu_lower for word in ['비빔밥','볶음밥','공깃밥','정식']):
        return 'rice'
    else:
        return 'others'

def get_store_type(store_name: str) -> str:
    if store_name in ["느티나무 셀프BBQ", "연회장"]:
        return 'Special_Occasion'
    elif store_name in ["화담숲주막", "화담숲카페"]:
        return 'Forest'
    elif store_name in ["담하", "미라시아"]:
        return 'Fine_Dining'
    elif store_name in ["카페테리아", "포레스트릿"]:
        return 'Casual'
    else:
        return 'Unique_Venue'

def get_weekend_sales_ratio(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy['영업일자'] = pd.to_datetime(df_copy['영업일자'])
    df_copy['요일'] = df_copy['영업일자'].dt.weekday
    weekend_df = df_copy[df_copy['요일'].isin([5,6])]
    weekday_df = df_copy[~df_copy['요일'].isin([5,6])]
    weekend_sales = weekend_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    weekday_sales = weekday_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    sales_ratio = (weekend_sales / weekday_sales).fillna(1.0).reset_index()
    sales_ratio.rename(columns={'매출수량':'weekend_sales_ratio'}, inplace=True)
    sales_ratio['weekend_sales_ratio'] = sales_ratio['weekend_sales_ratio'].replace([float('inf'), -float('inf')], 1.0)
    return sales_ratio

# ===== Correlation-based & DOW-strong features =====
def _safe_shift(a: pd.Series, n: int) -> pd.Series:
    return a.shift(n)

def _safe_rmean(a: pd.Series, w: int) -> pd.Series:
    return a.shift(1).rolling(window=w, min_periods=1).mean()

def build_corr_feature_bank(
    pivot: pd.DataFrame,              # index=date, columns=item, values=sales
    cutoff_date: pd.Timestamp,        # train_end_date (누수 방지)
    threshold: float = 0.5,
    topn: int = 5,
    lags: Tuple[int, ...] = (1, 7),
    rmeans: Tuple[int, ...] = (7, 14),
) -> Tuple[Dict[str, List[str]], Dict[str, pd.DataFrame]]:
    # train 구간만으로 상관 계산
    pivot_train = pivot.loc[:cutoff_date]
    corr = pivot_train.corr(method="pearson").fillna(0.0)

    corr_map: Dict[str, List[str]] = {}
    for tgt in corr.columns:
        partners_all = corr.index[(corr[tgt] >= threshold) & (corr.index != tgt)].tolist()
        # 상관값 기준 정렬 후 상위 N 제한
        partners_all = sorted(partners_all, key=lambda it: corr.loc[it, tgt], reverse=True)
        if topn and topn > 0:
            partners_all = partners_all[:topn]
        corr_map[tgt] = partners_all

    feat_bank: Dict[str, pd.DataFrame] = {}
    for tgt in pivot.columns:
        df_list = []
        for partner in corr_map.get(tgt, []):
            s = pivot[partner].astype(float)
            for L in lags:
                df_list.append(_safe_shift(s, L).rename(f"{partner}_lag{L}"))
            for W in rmeans:
                df_list.append(_safe_rmean(s, W).rename(f"{partner}_rmean{W}_lag1"))
        if df_list:
            fb = pd.concat(df_list, axis=1)
        else:
            fb = pd.DataFrame(index=pivot.index)
        feat_bank[tgt] = fb
    return corr_map, feat_bank

def compute_dow_strength_flags(df: pd.DataFrame,
                               date_col: str, item_col: str, target_col: str,
                               cutoff_date: pd.Timestamp,
                               ratio_threshold: float = 2.0,
                               min_support: int = 4) -> pd.DataFrame:
    tmp = df.copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.loc[tmp[date_col] <= cutoff_date]  # 누수 방지
    tmp["dow"] = tmp[date_col].dt.weekday

    overall = tmp.groupby(item_col)[target_col].mean()
    by_dow = tmp.groupby([item_col, "dow"])[target_col].agg(['mean','count']).unstack("dow")
    # 구조: columns MultiIndex [('mean',0..6), ('count',0..6)]
    mean_mat = by_dow['mean'].fillna(0.0)
    cnt_mat  = by_dow['count'].fillna(0.0)

    ratio = mean_mat.divide(overall, axis=0).fillna(0.0)
    strong = (ratio >= ratio_threshold) & (cnt_mat >= min_support)

    # 날짜 index × item 행렬 플래그 생성
    all_dates = pd.date_range(tmp[date_col].min(), df[date_col].max())
    items = tmp[item_col].unique().tolist()
    flag = pd.DataFrame(0, index=all_dates, columns=items, dtype=np.int8)
    for it in items:
        strong_dows = set(np.where(strong.loc[it].values)[0]) if it in strong.index else set()
        if not strong_dows:
            continue
        # 날짜별 dow가 strong이면 1
        dows = pd.Series(all_dates.weekday, index=all_dates)
        flag.loc[dows.isin(strong_dows), it] = 1
    return flag

# =====================
# Dataset
# =====================
class EnhancedNHiTSDataset(Dataset):
    """
    원본의 Dense 보강/캘린더피처/주말비율/인덱싱 로직 + (옵션) 상관 기반 lag/rmean, 요일강세 플래그
    """
    def __init__(self, cfg: EnhancedNHiTSConfig, df: pd.DataFrame):
        self.cfg = cfg
        self.df = df.copy()
        if cfg.date_col not in self.df.columns or cfg.item_col not in self.df.columns or cfg.target_col not in self.df.columns:
            raise ValueError("입력 데이터에 필요한 컬럼이 없습니다.")
        self.df[cfg.date_col] = pd.to_datetime(self.df[cfg.date_col])
        self.df[cfg.target_col] = pd.to_numeric(self.df[cfg.target_col], errors='coerce').fillna(0.0).clip(lower=0)

        # Dense dataset 보강 (모든 날짜×아이템)
        all_dates = pd.date_range(start=self.df[cfg.date_col].min(), end=self.df[cfg.date_col].max())
        all_items = self.df[cfg.item_col].unique()
        full_df = pd.MultiIndex.from_product([all_dates, all_items], names=[cfg.date_col, cfg.item_col]).to_frame(index=False)
        self.df = pd.merge(full_df, self.df, on=[cfg.date_col, cfg.item_col], how='left')
        self.df[cfg.target_col] = self.df[cfg.target_col].fillna(0)

        # 주말 상대 판매량 비율 병합
        weekend_ratio_df = get_weekend_sales_ratio(self.df)
        self.df = pd.merge(self.df, weekend_ratio_df, on=cfg.item_col, how='left')
        self.df['weekend_sales_ratio'].fillna(1.0, inplace=True)

        # 기본 피벗
        pivot = self.df.pivot(index=cfg.date_col, columns=cfg.item_col, values=cfg.target_col).sort_index()
        self.items = list(pivot.columns)
        self.dates = list(pivot.index)
        self.values = pivot.fillna(0.0).values.astype(np.float32)

        # 캘린더 피처
        holidays_set = set(pd.to_datetime(cfg.custom_holidays_list)) if cfg.custom_holidays_list else set()
        caldf = build_enhanced_features(self.dates, holidays_set)
        self.cal_feats = caldf.drop(columns=["date"]).values.astype(np.float32)
        self.cal_feat_names = [c for c in caldf.columns if c != "date"]

        # 인덱싱용 메타
        stores = [parse_store_name(it) for it in self.items]
        menus  = [parse_menu_name(it) for it in self.items]

        self.store2idx = {s: i for i, s in enumerate(sorted(set(stores)))}
        self.item_store_idx = np.array([self.store2idx[s] for s in stores], dtype=np.int64)
        self.n_stores = len(self.store2idx)

        self.cat2idx = {c: i for i, c in enumerate(sorted(set([get_menu_category(m) for m in menus])))}
        self.item_cat_idx = np.array([self.cat2idx[get_menu_category(m)] for m in menus], dtype=np.int64)
        self.n_categories = len(self.cat2idx)

        self.type2idx = {t: i for i, t in enumerate(sorted(set([get_store_type(s) for s in stores])))}
        self.item_type_idx = np.array([self.type2idx[get_store_type(s)] for s in stores], dtype=np.int64)
        self.n_types = len(self.type2idx)

        # ===== Extra features (옵션) =====
        cutoff = pd.to_datetime(cfg.train_end_date)
        self.extra_dim = 0
        self.extra_feats_per_item: List[np.ndarray] = []

        # (A) 요일 강세 플래그
        if cfg.USE_DOW_STRENGTH:
            dow_flag = compute_dow_strength_flags(
                self.df, cfg.date_col, cfg.item_col, cfg.target_col,
                cutoff_date=cutoff, ratio_threshold=cfg.DOW_RATIO_THRESHOLD, min_support=cfg.DOW_MIN_SUPPORT
            )
            dow_flag = dow_flag.reindex(index=pivot.index, columns=pivot.columns).fillna(0).astype(np.float32)
        else:
            dow_flag = pd.DataFrame(index=pivot.index, columns=pivot.columns, data=0.0, dtype=np.float32)

        # (B) 상관 기반 lag/rolling
        if cfg.USE_CORR_FEATURES:
            corr_map, feat_bank = build_corr_feature_bank(
                pivot=pivot, cutoff_date=cutoff,
                threshold=cfg.CORR_THRESHOLD, topn=cfg.CORR_TOPN,
                lags=cfg.CORR_LAGS, rmeans=cfg.CORR_RMEANS
            )
        else:
            corr_map, feat_bank = {}, {it: pd.DataFrame(index=pivot.index) for it in self.items}

        # 아이템별 extra DataFrame 구성 + 요일 강세 1채널 추가
        for it in self.items:
            fb = feat_bank.get(it, pd.DataFrame(index=pivot.index))
            fb = fb.copy()
            fb["_dow_strong"] = dow_flag[it] if it in dow_flag.columns else 0.0
            fb = fb.reindex(index=pivot.index).fillna(0.0).astype(np.float32)

            # 스케일링 옵션
            if fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "log1p":
                fb = np.log1p(fb)
            elif fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "zscore":
                mu = fb.mean(axis=0).replace(0.0, 0.0)
                std = fb.std(axis=0).replace(0.0, 1.0)
                fb = (fb - mu) / std

            self.extra_feats_per_item.append(fb.values)

        if self.extra_feats_per_item and self.extra_feats_per_item[0].shape[1] > 0:
            # 열 수를 모든 아이템에서 동일하게 맞춤
            min_dim = min(arr.shape[1] for arr in self.extra_feats_per_item)
            self.extra_feats_per_item = [arr[:, :min_dim] for arr in self.extra_feats_per_item]
            self.extra_dim = min_dim
        else:
            self.extra_dim = 0

        # 샘플 가중치
        sw = self.cfg.store_weights or {}
        self.sample_weights = np.array([sw.get(parse_store_name(it), 1.0) for it in self.items], dtype=np.float32)

        # 인덱스 생성
        self.indices: List[Tuple[int,int]] = []
        self.target_end_dates: List[pd.Timestamp] = []
        T = len(self.dates)
        Lx, Ly = cfg.in_len, cfg.out_len
        cutoff_date = pd.to_datetime(cfg.train_end_date)
        max_start = max(0, T - (Lx + Ly))
        for j in range(len(self.items)):
            for t0 in range(0, max_start + 1):
                end_date = self.dates[t0 + Lx + Ly - 1]
                if end_date <= cutoff_date:
                    self.indices.append((t0, j))
                    self.target_end_dates.append(end_date)

        self.target_end_dates = np.asarray(self.target_end_dates, dtype='datetime64[ns]')

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        cfg = self.cfg
        t0, j = self.indices[idx]
        Lx, Ly = cfg.in_len, cfg.out_len

        x = self.values[t0:t0 + Lx, j]
        y = self.values[t0 + Lx:t0 + Lx + Ly, j]

        if cfg.log1p:
            x_in = np.log1p(x); y_out = np.log1p(y)
        else:
            x_in, y_out = x.copy(), y.copy()

        past_cal = self.cal_feats[t0:t0 + Lx, :]
        fut_cal  = self.cal_feats[t0 + Lx:t0 + Lx + Ly, :]

        # 추가 피처
        if self.extra_dim > 0:
            ex_full = self.extra_feats_per_item[j]  # [T, E]
            past_ex = ex_full[t0:t0 + Lx, :]
            fut_ex  = np.zeros((Ly, self.extra_dim), dtype=np.float32)  # 미래는 0
        else:
            past_ex = np.zeros((Lx, 0), dtype=np.float32)
            fut_ex  = np.zeros((Ly, 0), dtype=np.float32)

        store_idx = self.item_store_idx[j]
        cat_idx   = self.item_cat_idx[j]
        type_idx  = self.item_type_idx[j]
        sample_w  = self.sample_weights[j]
        pos_mask  = (y > 0).astype(np.float32)

        zero_ratio = np.mean(x == 0)
        weekend_factor = np.mean(past_cal[:, 12])  # (참고) 예전 코드와 동일한 위치 피처 예시

        return {
            "x": torch.from_numpy(x_in).float(),
            "y": torch.from_numpy(y_out).float(),
            "past_cal": torch.from_numpy(past_cal).float(),
            "fut_cal": torch.from_numpy(fut_cal).float(),
            "past_ex": torch.from_numpy(past_ex).float(),
            "fut_ex": torch.from_numpy(fut_ex).float(),
            "store_idx": torch.tensor(store_idx, dtype=torch.long),
            "cat_idx": torch.tensor(cat_idx, dtype=torch.long),
            "type_idx": torch.tensor(type_idx, dtype=torch.long),
            "sample_w": torch.tensor(sample_w, dtype=torch.float32),
            "pos_mask": torch.from_numpy(pos_mask).float(),
            "zero_ratio": torch.tensor(zero_ratio, dtype=torch.float32),
            "weekend_factor": torch.tensor(weekend_factor, dtype=torch.float32),
            # 주말 비율은 필요 시 추가 가능
        }

def make_val_mask_by_week(dataset: 'EnhancedNHiTSDataset', end_date_str: str) -> np.ndarray:
    end_date = pd.to_datetime(end_date_str)
    start_date = end_date - pd.Timedelta(days=6)
    ted = dataset.target_end_dates
    return (ted >= start_date) & (ted <= end_date)

# =====================
# TCN (sequence head + meta)
# =====================
class _TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3, d=1, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(c_out, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(),
        )
        self.proj = nn.Conv1d(c_in, c_out, kernel_size=1) if c_in!=c_out else nn.Identity()
        self.norm = nn.LayerNorm(c_out)

    def forward(self, x):
        y = self.net(x) + self.proj(x)
        return self.norm(y.transpose(1,2)).transpose(1,2)

class TCNTiny(nn.Module):
    def __init__(self, cfg, in_len, out_len, cal_dim, n_stores, n_categories, n_types, C=128, depth=4, drop=0.1):
        super().__init__()
        self.out_len = out_len
        self.stem = nn.Conv1d(1, C, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([_TCNBlock(C, C, k=3, d=2**i, p=drop) for i in range(depth)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(C, out_len))

        self.store_emb = nn.Embedding(n_stores,64)
        self.cat_emb   = nn.Embedding(n_categories,32)
        self.type_emb  = nn.Embedding(n_types,16)
        self.cal_proj  = nn.Linear(cal_dim,128)

        self.meta_head = nn.Sequential(nn.Linear(64+32+16+128, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))
        self.prob_head = nn.Sequential(nn.Linear(64+32+16+128+1, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))

    def forward(self, x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                weekend_ratio=None, past_ex=None, fut_ex=None):
        # sequence path
        z = self.stem(x.unsqueeze(1))
        for blk in self.blocks:
            z = blk(z)
        seq_out = self.head(z)

        # extra concat
        if (past_ex is not None) and (fut_ex is not None) and (past_ex.numel() > 0):
            past_cal = torch.cat([past_cal, past_ex], dim=-1)
            fut_cal  = torch.cat([fut_cal,  fut_ex ], dim=-1)

        # meta path
        store = self.store_emb(store_idx); cat = self.cat_emb(cat_idx); typ = self.type_emb(type_idx)
        cal_all = torch.cat([past_cal, fut_cal], dim=1).mean(dim=1)  # [B, cal+extra]
        cal = self.cal_proj(cal_all)
        meta = torch.cat([store, cat, typ, cal], dim=-1)

        value_pred  = seq_out + self.meta_head(meta)
        prob_logits = self.prob_head(torch.cat([meta, x.mean(dim=1, keepdim=True)], dim=-1))
        return value_pred, prob_logits

# =====================
# Loss / EMA
# =====================
class UltraEnhancedHurdleLoss(nn.Module):
    def __init__(self, eps: float = 0.01, zero_weight: float = 0.01, lambda_bce: float = 0.15):
        super().__init__()
        self.eps = eps; self.zero_weight = zero_weight; self.lambda_bce = lambda_bce
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, v_pred_log, p_logits, y_true_log, pos_mask, sample_w):
        z = (pos_mask > 0).float()
        bce = self.bce(p_logits, z)

        yp_val = torch.expm1(v_pred_log).clamp_min(0.0)
        yt_val = torch.expm1(y_true_log).clamp_min(0.0)

        ultra_eps = torch.where(yt_val < 0.5, self.eps * 0.2,
                                torch.where(yt_val < 2.0, self.eps * 0.5, self.eps))
        denom = (torch.abs(yp_val) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_pos = 2.0 * torch.abs(yp_val - yt_val) / denom
        smape_pos = smape_pos * (yt_val > 0).float()

        p = torch.sigmoid(p_logits)
        y_hat = p * yp_val
        denom2 = (torch.abs(y_hat) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_all = 2.0 * torch.abs(y_hat - yt_val) / denom2

        ultra_zero_weight = torch.where(
            yt_val < 0.01, torch.full_like(yt_val, self.zero_weight * 0.1),
            torch.where(yt_val < 0.1, torch.full_like(yt_val, self.zero_weight * 0.3),
                        torch.where(yt_val < 1.0, torch.full_like(yt_val, self.zero_weight * 0.6), torch.ones_like(yt_val)))
        )
        smape_all = smape_all * ultra_zero_weight

        bce_s = bce.mean(dim=1); pos_s = smape_pos.mean(dim=1); all_s = smape_all.mean(dim=1)
        sample_loss = self.lambda_bce * bce_s + 0.4 * pos_s + 0.6 * all_s

        sw = sample_w.view(-1); wsum = sw.sum().clamp_min(1e-8)
        loss = (sample_loss * sw).sum() / wsum
        return loss, sample_loss.detach(), sw.detach()

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.shadow = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

    def update(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[name].mul_((self.decay)).add_(p.detach(), alpha=1-self.decay)

    def apply_to(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    p.copy_(self.shadow[name])

# =====================
# Trainer
# =====================
class GenericTrainer:
    def __init__(self, cfg: EnhancedNHiTSConfig, dataset: EnhancedNHiTSDataset, model: nn.Module,
                 epochs: int, batch_size: int, base_lr: float, max_lr: float, weight_decay: float):
        self.cfg = cfg; self.dataset = dataset; self.model = model.to(cfg.device)
        self.device = torch.device(cfg.device)
        self.criterion = UltraEnhancedHurdleLoss(cfg.eps_smape, cfg.zero_weight, cfg.hurdle_lambda)
        self.ema = EMA(self.model, decay=cfg.ema_decay)
        self.epochs = epochs; self.batch_size = batch_size
        self.base_lr = base_lr; self.max_lr = max_lr; self.weight_decay = weight_decay

    def make_loaders_from_mask(self, mask_val: np.ndarray):
        idx_all = np.arange(len(self.dataset))
        val_idx = idx_all[mask_val]; train_idx = idx_all[~mask_val]
        train_subset = torch.utils.data.Subset(self.dataset, train_idx)
        val_subset = torch.utils.data.Subset(self.dataset, val_idx)
        train_loader = DataLoader(
            train_subset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        val_loader = DataLoader(
            val_subset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        return train_loader, val_loader

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, use_ema: bool = True) -> float:
        if loader is None: return float('inf')
        self.model.eval()
        backup = None
        if use_ema:
            backup = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            self.ema.apply_to(self.model)

        total_loss, total_weight = 0.0, 0.0
        amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
            self.cfg.use_amp and torch.cuda.is_available() and self.device.type == "cuda"
        ) else torch.cuda.amp.autocast(enabled=False)

        with amp_ctx:
            for batch in loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                v_pred, p_logits = self.model(
                    x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                    weekend_ratio=None,
                    past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                    fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                )
                _, sample_loss, sw = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                total_loss += (sample_loss * sw).sum().item()
                total_weight += sw.sum().item()

        if use_ema and backup is not None:
            self.model.load_state_dict(backup); del backup
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        if total_weight <= 0: return float('inf')
        return total_loss / total_weight

    def train_with_loaders(self, train_loader: DataLoader, val_loader: DataLoader, model_name: str = "TCN"):
        optim = torch.optim.AdamW(self.model.parameters(), lr=self.base_lr, weight_decay=self.weight_decay, betas=(0.9, 0.999))
        steps_per_epoch = max(1, len(train_loader))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            optim, max_lr=self.max_lr, epochs=self.epochs,
            steps_per_epoch=steps_per_epoch, pct_start=0.05,
            div_factor=max(1e-8, self.max_lr / max(1e-8, self.base_lr))
        )
        patience = max(self.cfg.earlystop_patience_min, int(self.epochs * self.cfg.earlystop_patience_ratio))
        best_val = float("inf"); best_state = None; no_improve = 0

        for epoch in range(1, self.epochs+1):
            self.model.train()
            amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
                self.cfg.use_amp and torch.cuda.is_available() and self.device.type=="cuda"
            ) else torch.cuda.amp.autocast(enabled=False)
            running = []
            for batch in train_loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                with amp_ctx:
                    v_pred, p_logits = self.model(
                        x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                        weekend_ratio=None,
                        past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                        fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                    )
                    loss, _, _ = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                optim.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                optim.step(); sched.step(); self.ema.update(self.model)
                running.append(loss.detach().item())

            val_loss = self.evaluate(val_loader, use_ema=True)
            tr_mean = float(np.mean(running)) if running else float('nan')
            print(f"[{model_name}] [Epoch {epoch:03d}] train_loss: {tr_mean:.5f}  val_loss: {val_loss:.5f}")

            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch} (patience={patience})")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state, strict=True)
        return self.model, best_val

# =====================
# Build helper (TCN만)
# =====================
def build_tcn(cfg: EnhancedNHiTSConfig, ds: EnhancedNHiTSDataset, C=128, depth=4, drop=0.1):
    cal_dim = int(ds.cal_feats.shape[1] + getattr(ds, "extra_dim", 0))
    return TCNTiny(cfg, cfg.in_len, cfg.out_len, cal_dim, ds.n_stores, ds.n_categories, ds.n_types, C=C, depth=depth, drop=drop)

# =====================
# Main
# =====================
if __name__ == "__main__":
    cfg = EnhancedNHiTSConfig()
    if cfg.store_weights is None: cfg.store_weights = DEFAULT_STORE_WEIGHTS
    if cfg.custom_holidays_list is None: cfg.custom_holidays_list = DEFAULT_CUSTOM_HOLIDAYS
    set_seed(cfg.seed)

    if not os.path.exists(cfg.train_csv):
        raise FileNotFoundError(f"학습 파일을 찾을 수 없습니다: {cfg.train_csv}")

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    # Load train
    train_df = pd.read_csv(cfg.train_csv)
    ds = EnhancedNHiTSDataset(cfg, train_df)

    # 정보 출력
    print(f"Base cal_dim = {ds.cal_feats.shape[1]}, extra_dim = {ds.extra_dim}, USE_CORR={cfg.USE_CORR_FEATURES}, USE_DOW={cfg.USE_DOW_STRENGTH}")

    # Validation: 마지막 주(예시)로 마스크
    cv_end = "2024-06-14"
    mask_val = make_val_mask_by_week(ds, cv_end)

    # Build TCN
    model = build_tcn(cfg, ds, C=128, depth=4, drop=0.1)

    # Train
    trainer = GenericTrainer(cfg, ds, model,
                             epochs=cfg.EPOCHS_FULL, batch_size=cfg.BATCH_FULL,
                             base_lr=cfg.BASE_LR_FULL, max_lr=cfg.MAX_LR_FULL, weight_decay=cfg.WD_FULL)
    train_loader, val_loader = trainer.make_loaders_from_mask(mask_val)
    model, best_val = trainer.train_with_loaders(train_loader, val_loader, model_name="TCN")

    # Save checkpoint
    ckpt_path = os.path.join(cfg.checkpoint_dir, "TCN_best_fold.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Saved best checkpoint to {ckpt_path} (val_loss={best_val:.6f})")


Base cal_dim = 32, extra_dim = 1, USE_CORR=True, USE_DOW=False
[TCN] [Epoch 001] train_loss: 0.40843  val_loss: 0.86754
[TCN] [Epoch 002] train_loss: 0.35979  val_loss: 0.78933
[TCN] [Epoch 003] train_loss: 0.34844  val_loss: 0.73855
[TCN] [Epoch 004] train_loss: 0.34090  val_loss: 0.71488
[TCN] [Epoch 005] train_loss: 0.33518  val_loss: 0.71247
[TCN] [Epoch 006] train_loss: 0.32952  val_loss: 0.71898
[TCN] [Epoch 007] train_loss: 0.32524  val_loss: 0.72469
[TCN] [Epoch 008] train_loss: 0.31994  val_loss: 0.72463
[TCN] [Epoch 009] train_loss: 0.31634  val_loss: 0.71394
[TCN] [Epoch 010] train_loss: 0.31292  val_loss: 0.69454
[TCN] [Epoch 011] train_loss: 0.30933  val_loss: 0.66940
[TCN] [Epoch 012] train_loss: 0.30642  val_loss: 0.64197
[TCN] [Epoch 013] train_loss: 0.30301  val_loss: 0.61419
[TCN] [Epoch 014] train_loss: 0.30054  val_loss: 0.58901
[TCN] [Epoch 015] train_loss: 0.29804  val_loss: 0.56640
[TCN] [Epoch 016] train_loss: 0.29533  val_loss: 0.54532
[TCN] [Epoch 017] train_l

# 요일 강세 피처 도입
[TCN] [Epoch 050] train_loss: 0.25230  val_loss: 0.37629


In [4]:
# tcn_only_with_corr_and_dow.py
# -*- coding: utf-8 -*-
import os, gc, math, random, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# =====================
# Config
# =====================
@dataclass
class EnhancedNHiTSConfig:
    # Paths
    train_csv: str = "/content/drive/MyDrive/data/train/train_original.csv"
    test_glob: str = "/content/drive/MyDrive/data/test/*.csv"
    submission_template_csv: str = "/content/drive/MyDrive/data/sample_submission.csv"
    out_submission_csv: str = "/content/drive/MyDrive/data/0823_submission.csv"
    checkpoint_dir: str = "/content/drive/MyDrive/data/checkpoint/enhanced_checkpoints"

    # Columns
    date_col: str = "영업일자"
    item_col: str = "영업장명_메뉴명"
    target_col: str = "매출수량"

    # Window
    in_len: int = 28
    out_len: int = 7

    # Train filtering
    train_end_date: str = "2024-06-15"

    # General
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log1p: bool = True

    # Dataloaders
    num_workers: int = 4
    pin_memory: bool = True
    persistent_workers: bool = False

    # Loss
    eps_smape: float = 0.01
    zero_weight: float = 0.01
    hurdle_lambda: float = 0.15

    # AMP/EMA
    use_amp: bool = True
    use_compile: bool = False
    ema_decay: float = 0.9998

    # Train hyperparams
    EPOCHS_FULL: int = 100
    BATCH_FULL: int = 256
    BASE_LR_FULL: float = 6e-4
    MAX_LR_FULL: float = 1.5e-3
    WD_FULL: float = 6e-4

    # Early stop / grad clip
    grad_clip: float = 0.5
    earlystop_patience_ratio: float = 0.12
    earlystop_patience_min: int = 8

    # Optional weights/holidays (채워지지 않으면 기본 사용)
    store_weights: Optional[Dict[str, float]] = None
    custom_holidays_list: Optional[List[str]] = None

    # ====== NEW: Feature toggles & params ======
    # Corr-based features
    USE_CORR_FEATURES: bool = False
    CORR_THRESHOLD: float = 0.5     # 상관 임계치
    CORR_TOPN: int = 5              # 메뉴당 상위 파트너 N개 제한 (0 = 제한 없음)
    CORR_LAGS: Tuple[int, ...] = (1, 7)
    CORR_RMEANS: Tuple[int, ...] = (7, 14)  # 모두 shift=1

    # Day-of-week strength
    USE_DOW_STRENGTH: bool = True
    DOW_RATIO_THRESHOLD: float = 2.0    # 평균 대비 2배 이상
    DOW_MIN_SUPPORT: int = 4            # 요일 평균 계산에 필요한 최소 관측일 수

    # Scaling extra features: "none" | "log1p" | "zscore"
    EXTRA_FEAT_SCALING: str = "none"


DEFAULT_STORE_WEIGHTS = {
    "미라시아": 1, "담하": 1, "연회장": 1, "라그로타": 1,
    "느티나무 셀프BBQ": 1, "화담숲주막": 1, "카페테리아": 1,
    "화담숲카페": 1, "포레스트릿": 1,
}
DEFAULT_CUSTOM_HOLIDAYS = [
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-01',
    '2023-05-05','2023-05-27','2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30',
    '2023-10-02','2023-10-03','2023-10-09','2023-12-25',
    '2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01','2024-04-10',
    '2024-05-01','2024-05-05','2024-05-06','2024-05-15','2024-06-06','2024-08-15','2024-09-16',
    '2024-09-17','2024-09-18','2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    '2025-01-01','2025-01-28','2025-01-29','2025-01-30','2025-03-01','2025-03-03','2025-05-01',
    '2025-05-05','2025-05-06','2025-06-06','2025-08-15'
]

# =====================
# Utils
# =====================
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sine_cosine_encoding(value: float, max_val: float):
    return math.sin(2 * math.pi * value / max_val), math.cos(2 * math.pi * value / max_val)

def parse_store_name(item_full: str) -> str:
    return item_full.split("_")[0]

def parse_menu_name(item_full: str) -> str:
    return item_full.split("_", 1)[1] if "_" in item_full else item_full

def build_enhanced_features(dates: List[pd.Timestamp], holidays_set: set,
                            store_names: Optional[List[str]] = None) -> pd.DataFrame:
    df = pd.DataFrame({"date": dates})
    df["dow"] = df["date"].dt.weekday
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["week"] = df["date"].apply(lambda d: int(d.isocalendar().week))
    df["quarter"] = df["date"].dt.quarter
    df["is_friday"] = (df["dow"] == 4).astype(int)
    df["is_saturday"] = (df["dow"] == 5).astype(int)
    df["is_sunday"] = (df["dow"] == 6).astype(int)
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["is_holiday"] = df["date"].isin(holidays_set).astype(int)
    df["is_off"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)
    df["tomorrow"] = df["date"] + pd.Timedelta(days=1)
    df["yesterday"] = df["date"] - pd.Timedelta(days=1)
    df["is_before_holiday"] = df["tomorrow"].isin(holidays_set).astype(int)
    df["is_after_holiday"] = df["yesterday"].isin(holidays_set).astype(int)
    df["is_month_start"] = (df["day"] <= 3).astype(int)
    df["is_month_end"] = (df["day"] >= 28).astype(int)
    df["is_spring"] = df["month"].isin([4, 5, 6]).astype(int)
    df["is_summer"] = df["month"].isin([7, 8]).astype(int)
    df["is_autumn"] = df["month"].isin([9, 10, 11]).astype(int)
    df["is_winter"] = df["month"].isin([12, 1, 2, 3]).astype(int)
    df["is_summer_vacation"] = df["month"].isin([7, 8]).astype(int)
    df["is_winter_vacation"] = df["month"].isin([12, 1, 2]).astype(int)

    # 날짜 기반 정기 휴무일 (원본 로직 요약)
    df['is_regular_holiday'] = 0
    df.loc[((df['date'].dt.month.isin(range(2, 12))) & (df['date'].dt.year == 2023) & (df['dow'] == 0)) |
           ((df['date'].dt.month.isin(range(3, 7))) & (df['date'].dt.year == 2024) & (df['dow'] == 0)) |
           (df['date'].isin(['2023-03-01'])) |
           (df['date'].isin(['2023-09-01','2023-09-02','2023-09-03']) & (df['dow'].isin([0,1,2]))) |
           (df['date'].isin(['2024-03-01'])) |
           ((df['date'] >= '2023-05-01') & (df['dow'].isin([0,1,2,3]))) |
           (df['date'].dt.month.isin([12,1,2,3]) & df['date'].dt.year.isin([2023,2024,2025])) |
           ((~df['date'].dt.month.isin([12,1,2,3])) & (df['dow'] == 0)), 'is_regular_holiday'] = 1

    df[["month_sin", "month_cos"]] = df["month"].apply(lambda m: pd.Series(sine_cosine_encoding(m, 12)))
    df[["dow_sin", "dow_cos"]] = df["dow"].apply(lambda d: pd.Series(sine_cosine_encoding(d, 7)))
    df[["doy_sin", "doy_cos"]] = df["date"].apply(lambda d: pd.Series(sine_cosine_encoding(d.dayofyear, 365)))
    df[["week_sin", "week_cos"]] = df["week"].apply(lambda w: pd.Series(sine_cosine_encoding(w, 52)))
    df[["quarter_sin", "quarter_cos"]] = df["quarter"].apply(lambda q: pd.Series(sine_cosine_encoding(q, 4)))
    return df.drop(columns=["tomorrow", "yesterday"])

def get_menu_category(menu_name: str) -> str:
    menu_lower = menu_name.lower()
    if any(word in menu_lower for word in ['막걸리','소주','맥주','와인','참이슬','처음처럼','카스','하이네켄','버드와이저','스텔라']):
        return 'alcohol'
    elif any(word in menu_lower for word in ['찌개','탕','국밥','라면','해장국','갈비탕']):
        return 'hot_food'
    elif any(word in menu_lower for word in ['삼겹','갈비','목살','bbq','구이','불고기']):
        return 'bbq'
    elif any(word in menu_lower for word in ['아이스크림','식혜','콜라','스프라이트','에이드']):
        return 'dessert_drink'
    elif any(word in menu_lower for word in ['아메리카노','라떼','커피']):
        return 'coffee'
    elif any(word in menu_lower for word in ['냉면','파스타','스파게티','면','우동']):
        return 'noodles'
    elif any(word in menu_lower for word in ['비빔밥','볶음밥','공깃밥','정식']):
        return 'rice'
    else:
        return 'others'

def get_store_type(store_name: str) -> str:
    if store_name in ["느티나무 셀프BBQ", "연회장"]:
        return 'Special_Occasion'
    elif store_name in ["화담숲주막", "화담숲카페"]:
        return 'Forest'
    elif store_name in ["담하", "미라시아"]:
        return 'Fine_Dining'
    elif store_name in ["카페테리아", "포레스트릿"]:
        return 'Casual'
    else:
        return 'Unique_Venue'

def get_weekend_sales_ratio(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy['영업일자'] = pd.to_datetime(df_copy['영업일자'])
    df_copy['요일'] = df_copy['영업일자'].dt.weekday
    weekend_df = df_copy[df_copy['요일'].isin([5,6])]
    weekday_df = df_copy[~df_copy['요일'].isin([5,6])]
    weekend_sales = weekend_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    weekday_sales = weekday_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    sales_ratio = (weekend_sales / weekday_sales).fillna(1.0).reset_index()
    sales_ratio.rename(columns={'매출수량':'weekend_sales_ratio'}, inplace=True)
    sales_ratio['weekend_sales_ratio'] = sales_ratio['weekend_sales_ratio'].replace([float('inf'), -float('inf')], 1.0)
    return sales_ratio

# ===== Correlation-based & DOW-strong features =====
def _safe_shift(a: pd.Series, n: int) -> pd.Series:
    return a.shift(n)

def _safe_rmean(a: pd.Series, w: int) -> pd.Series:
    return a.shift(1).rolling(window=w, min_periods=1).mean()

def build_corr_feature_bank(
    pivot: pd.DataFrame,              # index=date, columns=item, values=sales
    cutoff_date: pd.Timestamp,        # train_end_date (누수 방지)
    threshold: float = 0.5,
    topn: int = 5,
    lags: Tuple[int, ...] = (1, 7),
    rmeans: Tuple[int, ...] = (7, 14),
) -> Tuple[Dict[str, List[str]], Dict[str, pd.DataFrame]]:
    # train 구간만으로 상관 계산
    pivot_train = pivot.loc[:cutoff_date]
    corr = pivot_train.corr(method="pearson").fillna(0.0)

    corr_map: Dict[str, List[str]] = {}
    for tgt in corr.columns:
        partners_all = corr.index[(corr[tgt] >= threshold) & (corr.index != tgt)].tolist()
        # 상관값 기준 정렬 후 상위 N 제한
        partners_all = sorted(partners_all, key=lambda it: corr.loc[it, tgt], reverse=True)
        if topn and topn > 0:
            partners_all = partners_all[:topn]
        corr_map[tgt] = partners_all

    feat_bank: Dict[str, pd.DataFrame] = {}
    for tgt in pivot.columns:
        df_list = []
        for partner in corr_map.get(tgt, []):
            s = pivot[partner].astype(float)
            for L in lags:
                df_list.append(_safe_shift(s, L).rename(f"{partner}_lag{L}"))
            for W in rmeans:
                df_list.append(_safe_rmean(s, W).rename(f"{partner}_rmean{W}_lag1"))
        if df_list:
            fb = pd.concat(df_list, axis=1)
        else:
            fb = pd.DataFrame(index=pivot.index)
        feat_bank[tgt] = fb
    return corr_map, feat_bank

def compute_dow_strength_flags(df: pd.DataFrame,
                               date_col: str, item_col: str, target_col: str,
                               cutoff_date: pd.Timestamp,
                               ratio_threshold: float = 2.0,
                               min_support: int = 4) -> pd.DataFrame:
    tmp = df.copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.loc[tmp[date_col] <= cutoff_date]  # 누수 방지
    tmp["dow"] = tmp[date_col].dt.weekday

    overall = tmp.groupby(item_col)[target_col].mean()
    by_dow = tmp.groupby([item_col, "dow"])[target_col].agg(['mean','count']).unstack("dow")
    # 구조: columns MultiIndex [('mean',0..6), ('count',0..6)]
    mean_mat = by_dow['mean'].fillna(0.0)
    cnt_mat  = by_dow['count'].fillna(0.0)

    ratio = mean_mat.divide(overall, axis=0).fillna(0.0)
    strong = (ratio >= ratio_threshold) & (cnt_mat >= min_support)

    # 날짜 index × item 행렬 플래그 생성
    all_dates = pd.date_range(tmp[date_col].min(), df[date_col].max())
    items = tmp[item_col].unique().tolist()
    flag = pd.DataFrame(0, index=all_dates, columns=items, dtype=np.int8)
    for it in items:
        strong_dows = set(np.where(strong.loc[it].values)[0]) if it in strong.index else set()
        if not strong_dows:
            continue
        # 날짜별 dow가 strong이면 1
        dows = pd.Series(all_dates.weekday, index=all_dates)
        flag.loc[dows.isin(strong_dows), it] = 1
    return flag

# =====================
# Dataset
# =====================
class EnhancedNHiTSDataset(Dataset):
    """
    원본의 Dense 보강/캘린더피처/주말비율/인덱싱 로직 + (옵션) 상관 기반 lag/rmean, 요일강세 플래그
    """
    def __init__(self, cfg: EnhancedNHiTSConfig, df: pd.DataFrame):
        self.cfg = cfg
        self.df = df.copy()
        if cfg.date_col not in self.df.columns or cfg.item_col not in self.df.columns or cfg.target_col not in self.df.columns:
            raise ValueError("입력 데이터에 필요한 컬럼이 없습니다.")
        self.df[cfg.date_col] = pd.to_datetime(self.df[cfg.date_col])
        self.df[cfg.target_col] = pd.to_numeric(self.df[cfg.target_col], errors='coerce').fillna(0.0).clip(lower=0)

        # Dense dataset 보강 (모든 날짜×아이템)
        all_dates = pd.date_range(start=self.df[cfg.date_col].min(), end=self.df[cfg.date_col].max())
        all_items = self.df[cfg.item_col].unique()
        full_df = pd.MultiIndex.from_product([all_dates, all_items], names=[cfg.date_col, cfg.item_col]).to_frame(index=False)
        self.df = pd.merge(full_df, self.df, on=[cfg.date_col, cfg.item_col], how='left')
        self.df[cfg.target_col] = self.df[cfg.target_col].fillna(0)

        # 주말 상대 판매량 비율 병합
        weekend_ratio_df = get_weekend_sales_ratio(self.df)
        self.df = pd.merge(self.df, weekend_ratio_df, on=cfg.item_col, how='left')
        self.df['weekend_sales_ratio'].fillna(1.0, inplace=True)

        # 기본 피벗
        pivot = self.df.pivot(index=cfg.date_col, columns=cfg.item_col, values=cfg.target_col).sort_index()
        self.items = list(pivot.columns)
        self.dates = list(pivot.index)
        self.values = pivot.fillna(0.0).values.astype(np.float32)

        # 캘린더 피처
        holidays_set = set(pd.to_datetime(cfg.custom_holidays_list)) if cfg.custom_holidays_list else set()
        caldf = build_enhanced_features(self.dates, holidays_set)
        self.cal_feats = caldf.drop(columns=["date"]).values.astype(np.float32)
        self.cal_feat_names = [c for c in caldf.columns if c != "date"]

        # 인덱싱용 메타
        stores = [parse_store_name(it) for it in self.items]
        menus  = [parse_menu_name(it) for it in self.items]

        self.store2idx = {s: i for i, s in enumerate(sorted(set(stores)))}
        self.item_store_idx = np.array([self.store2idx[s] for s in stores], dtype=np.int64)
        self.n_stores = len(self.store2idx)

        self.cat2idx = {c: i for i, c in enumerate(sorted(set([get_menu_category(m) for m in menus])))}
        self.item_cat_idx = np.array([self.cat2idx[get_menu_category(m)] for m in menus], dtype=np.int64)
        self.n_categories = len(self.cat2idx)

        self.type2idx = {t: i for i, t in enumerate(sorted(set([get_store_type(s) for s in stores])))}
        self.item_type_idx = np.array([self.type2idx[get_store_type(s)] for s in stores], dtype=np.int64)
        self.n_types = len(self.type2idx)

        # ===== Extra features (옵션) =====
        cutoff = pd.to_datetime(cfg.train_end_date)
        self.extra_dim = 0
        self.extra_feats_per_item: List[np.ndarray] = []

        # (A) 요일 강세 플래그
        if cfg.USE_DOW_STRENGTH:
            dow_flag = compute_dow_strength_flags(
                self.df, cfg.date_col, cfg.item_col, cfg.target_col,
                cutoff_date=cutoff, ratio_threshold=cfg.DOW_RATIO_THRESHOLD, min_support=cfg.DOW_MIN_SUPPORT
            )
            dow_flag = dow_flag.reindex(index=pivot.index, columns=pivot.columns).fillna(0).astype(np.float32)
        else:
            dow_flag = pd.DataFrame(index=pivot.index, columns=pivot.columns, data=0.0, dtype=np.float32)

        # (B) 상관 기반 lag/rolling
        if cfg.USE_CORR_FEATURES:
            corr_map, feat_bank = build_corr_feature_bank(
                pivot=pivot, cutoff_date=cutoff,
                threshold=cfg.CORR_THRESHOLD, topn=cfg.CORR_TOPN,
                lags=cfg.CORR_LAGS, rmeans=cfg.CORR_RMEANS
            )
        else:
            corr_map, feat_bank = {}, {it: pd.DataFrame(index=pivot.index) for it in self.items}

        # 아이템별 extra DataFrame 구성 + 요일 강세 1채널 추가
        for it in self.items:
            fb = feat_bank.get(it, pd.DataFrame(index=pivot.index))
            fb = fb.copy()
            fb["_dow_strong"] = dow_flag[it] if it in dow_flag.columns else 0.0
            fb = fb.reindex(index=pivot.index).fillna(0.0).astype(np.float32)

            # 스케일링 옵션
            if fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "log1p":
                fb = np.log1p(fb)
            elif fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "zscore":
                mu = fb.mean(axis=0).replace(0.0, 0.0)
                std = fb.std(axis=0).replace(0.0, 1.0)
                fb = (fb - mu) / std

            self.extra_feats_per_item.append(fb.values)

        if self.extra_feats_per_item and self.extra_feats_per_item[0].shape[1] > 0:
            # 열 수를 모든 아이템에서 동일하게 맞춤
            min_dim = min(arr.shape[1] for arr in self.extra_feats_per_item)
            self.extra_feats_per_item = [arr[:, :min_dim] for arr in self.extra_feats_per_item]
            self.extra_dim = min_dim
        else:
            self.extra_dim = 0

        # 샘플 가중치
        sw = self.cfg.store_weights or {}
        self.sample_weights = np.array([sw.get(parse_store_name(it), 1.0) for it in self.items], dtype=np.float32)

        # 인덱스 생성
        self.indices: List[Tuple[int,int]] = []
        self.target_end_dates: List[pd.Timestamp] = []
        T = len(self.dates)
        Lx, Ly = cfg.in_len, cfg.out_len
        cutoff_date = pd.to_datetime(cfg.train_end_date)
        max_start = max(0, T - (Lx + Ly))
        for j in range(len(self.items)):
            for t0 in range(0, max_start + 1):
                end_date = self.dates[t0 + Lx + Ly - 1]
                if end_date <= cutoff_date:
                    self.indices.append((t0, j))
                    self.target_end_dates.append(end_date)

        self.target_end_dates = np.asarray(self.target_end_dates, dtype='datetime64[ns]')

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        cfg = self.cfg
        t0, j = self.indices[idx]
        Lx, Ly = cfg.in_len, cfg.out_len

        x = self.values[t0:t0 + Lx, j]
        y = self.values[t0 + Lx:t0 + Lx + Ly, j]

        if cfg.log1p:
            x_in = np.log1p(x); y_out = np.log1p(y)
        else:
            x_in, y_out = x.copy(), y.copy()

        past_cal = self.cal_feats[t0:t0 + Lx, :]
        fut_cal  = self.cal_feats[t0 + Lx:t0 + Lx + Ly, :]

        # 추가 피처
        if self.extra_dim > 0:
            ex_full = self.extra_feats_per_item[j]  # [T, E]
            past_ex = ex_full[t0:t0 + Lx, :]
            fut_ex  = np.zeros((Ly, self.extra_dim), dtype=np.float32)  # 미래는 0
        else:
            past_ex = np.zeros((Lx, 0), dtype=np.float32)
            fut_ex  = np.zeros((Ly, 0), dtype=np.float32)

        store_idx = self.item_store_idx[j]
        cat_idx   = self.item_cat_idx[j]
        type_idx  = self.item_type_idx[j]
        sample_w  = self.sample_weights[j]
        pos_mask  = (y > 0).astype(np.float32)

        zero_ratio = np.mean(x == 0)
        weekend_factor = np.mean(past_cal[:, 12])  # (참고) 예전 코드와 동일한 위치 피처 예시

        return {
            "x": torch.from_numpy(x_in).float(),
            "y": torch.from_numpy(y_out).float(),
            "past_cal": torch.from_numpy(past_cal).float(),
            "fut_cal": torch.from_numpy(fut_cal).float(),
            "past_ex": torch.from_numpy(past_ex).float(),
            "fut_ex": torch.from_numpy(fut_ex).float(),
            "store_idx": torch.tensor(store_idx, dtype=torch.long),
            "cat_idx": torch.tensor(cat_idx, dtype=torch.long),
            "type_idx": torch.tensor(type_idx, dtype=torch.long),
            "sample_w": torch.tensor(sample_w, dtype=torch.float32),
            "pos_mask": torch.from_numpy(pos_mask).float(),
            "zero_ratio": torch.tensor(zero_ratio, dtype=torch.float32),
            "weekend_factor": torch.tensor(weekend_factor, dtype=torch.float32),
            # 주말 비율은 필요 시 추가 가능
        }

def make_val_mask_by_week(dataset: 'EnhancedNHiTSDataset', end_date_str: str) -> np.ndarray:
    end_date = pd.to_datetime(end_date_str)
    start_date = end_date - pd.Timedelta(days=6)
    ted = dataset.target_end_dates
    return (ted >= start_date) & (ted <= end_date)

# =====================
# TCN (sequence head + meta)
# =====================
class _TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3, d=1, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(c_out, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(),
        )
        self.proj = nn.Conv1d(c_in, c_out, kernel_size=1) if c_in!=c_out else nn.Identity()
        self.norm = nn.LayerNorm(c_out)

    def forward(self, x):
        y = self.net(x) + self.proj(x)
        return self.norm(y.transpose(1,2)).transpose(1,2)

class TCNTiny(nn.Module):
    def __init__(self, cfg, in_len, out_len, cal_dim, n_stores, n_categories, n_types, C=128, depth=4, drop=0.1):
        super().__init__()
        self.out_len = out_len
        self.stem = nn.Conv1d(1, C, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([_TCNBlock(C, C, k=3, d=2**i, p=drop) for i in range(depth)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(C, out_len))

        self.store_emb = nn.Embedding(n_stores,64)
        self.cat_emb   = nn.Embedding(n_categories,32)
        self.type_emb  = nn.Embedding(n_types,16)
        self.cal_proj  = nn.Linear(cal_dim,128)

        self.meta_head = nn.Sequential(nn.Linear(64+32+16+128, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))
        self.prob_head = nn.Sequential(nn.Linear(64+32+16+128+1, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))

    def forward(self, x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                weekend_ratio=None, past_ex=None, fut_ex=None):
        # sequence path
        z = self.stem(x.unsqueeze(1))
        for blk in self.blocks:
            z = blk(z)
        seq_out = self.head(z)

        # extra concat
        if (past_ex is not None) and (fut_ex is not None) and (past_ex.numel() > 0):
            past_cal = torch.cat([past_cal, past_ex], dim=-1)
            fut_cal  = torch.cat([fut_cal,  fut_ex ], dim=-1)

        # meta path
        store = self.store_emb(store_idx); cat = self.cat_emb(cat_idx); typ = self.type_emb(type_idx)
        cal_all = torch.cat([past_cal, fut_cal], dim=1).mean(dim=1)  # [B, cal+extra]
        cal = self.cal_proj(cal_all)
        meta = torch.cat([store, cat, typ, cal], dim=-1)

        value_pred  = seq_out + self.meta_head(meta)
        prob_logits = self.prob_head(torch.cat([meta, x.mean(dim=1, keepdim=True)], dim=-1))
        return value_pred, prob_logits

# =====================
# Loss / EMA
# =====================
class UltraEnhancedHurdleLoss(nn.Module):
    def __init__(self, eps: float = 0.01, zero_weight: float = 0.01, lambda_bce: float = 0.15):
        super().__init__()
        self.eps = eps; self.zero_weight = zero_weight; self.lambda_bce = lambda_bce
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, v_pred_log, p_logits, y_true_log, pos_mask, sample_w):
        z = (pos_mask > 0).float()
        bce = self.bce(p_logits, z)

        yp_val = torch.expm1(v_pred_log).clamp_min(0.0)
        yt_val = torch.expm1(y_true_log).clamp_min(0.0)

        ultra_eps = torch.where(yt_val < 0.5, self.eps * 0.2,
                                torch.where(yt_val < 2.0, self.eps * 0.5, self.eps))
        denom = (torch.abs(yp_val) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_pos = 2.0 * torch.abs(yp_val - yt_val) / denom
        smape_pos = smape_pos * (yt_val > 0).float()

        p = torch.sigmoid(p_logits)
        y_hat = p * yp_val
        denom2 = (torch.abs(y_hat) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_all = 2.0 * torch.abs(y_hat - yt_val) / denom2

        ultra_zero_weight = torch.where(
            yt_val < 0.01, torch.full_like(yt_val, self.zero_weight * 0.1),
            torch.where(yt_val < 0.1, torch.full_like(yt_val, self.zero_weight * 0.3),
                        torch.where(yt_val < 1.0, torch.full_like(yt_val, self.zero_weight * 0.6), torch.ones_like(yt_val)))
        )
        smape_all = smape_all * ultra_zero_weight

        bce_s = bce.mean(dim=1); pos_s = smape_pos.mean(dim=1); all_s = smape_all.mean(dim=1)
        sample_loss = self.lambda_bce * bce_s + 0.4 * pos_s + 0.6 * all_s

        sw = sample_w.view(-1); wsum = sw.sum().clamp_min(1e-8)
        loss = (sample_loss * sw).sum() / wsum
        return loss, sample_loss.detach(), sw.detach()

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.shadow = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

    def update(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[name].mul_((self.decay)).add_(p.detach(), alpha=1-self.decay)

    def apply_to(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    p.copy_(self.shadow[name])

# =====================
# Trainer
# =====================
class GenericTrainer:
    def __init__(self, cfg: EnhancedNHiTSConfig, dataset: EnhancedNHiTSDataset, model: nn.Module,
                 epochs: int, batch_size: int, base_lr: float, max_lr: float, weight_decay: float):
        self.cfg = cfg; self.dataset = dataset; self.model = model.to(cfg.device)
        self.device = torch.device(cfg.device)
        self.criterion = UltraEnhancedHurdleLoss(cfg.eps_smape, cfg.zero_weight, cfg.hurdle_lambda)
        self.ema = EMA(self.model, decay=cfg.ema_decay)
        self.epochs = epochs; self.batch_size = batch_size
        self.base_lr = base_lr; self.max_lr = max_lr; self.weight_decay = weight_decay

    def make_loaders_from_mask(self, mask_val: np.ndarray):
        idx_all = np.arange(len(self.dataset))
        val_idx = idx_all[mask_val]; train_idx = idx_all[~mask_val]
        train_subset = torch.utils.data.Subset(self.dataset, train_idx)
        val_subset = torch.utils.data.Subset(self.dataset, val_idx)
        train_loader = DataLoader(
            train_subset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        val_loader = DataLoader(
            val_subset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        return train_loader, val_loader

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, use_ema: bool = True) -> float:
        if loader is None: return float('inf')
        self.model.eval()
        backup = None
        if use_ema:
            backup = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            self.ema.apply_to(self.model)

        total_loss, total_weight = 0.0, 0.0
        amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
            self.cfg.use_amp and torch.cuda.is_available() and self.device.type == "cuda"
        ) else torch.cuda.amp.autocast(enabled=False)

        with amp_ctx:
            for batch in loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                v_pred, p_logits = self.model(
                    x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                    weekend_ratio=None,
                    past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                    fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                )
                _, sample_loss, sw = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                total_loss += (sample_loss * sw).sum().item()
                total_weight += sw.sum().item()

        if use_ema and backup is not None:
            self.model.load_state_dict(backup); del backup
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        if total_weight <= 0: return float('inf')
        return total_loss / total_weight

    def train_with_loaders(self, train_loader: DataLoader, val_loader: DataLoader, model_name: str = "TCN"):
        optim = torch.optim.AdamW(self.model.parameters(), lr=self.base_lr, weight_decay=self.weight_decay, betas=(0.9, 0.999))
        steps_per_epoch = max(1, len(train_loader))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            optim, max_lr=self.max_lr, epochs=self.epochs,
            steps_per_epoch=steps_per_epoch, pct_start=0.05,
            div_factor=max(1e-8, self.max_lr / max(1e-8, self.base_lr))
        )
        patience = max(self.cfg.earlystop_patience_min, int(self.epochs * self.cfg.earlystop_patience_ratio))
        best_val = float("inf"); best_state = None; no_improve = 0

        for epoch in range(1, self.epochs+1):
            self.model.train()
            amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
                self.cfg.use_amp and torch.cuda.is_available() and self.device.type=="cuda"
            ) else torch.cuda.amp.autocast(enabled=False)
            running = []
            for batch in train_loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                with amp_ctx:
                    v_pred, p_logits = self.model(
                        x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                        weekend_ratio=None,
                        past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                        fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                    )
                    loss, _, _ = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                optim.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                optim.step(); sched.step(); self.ema.update(self.model)
                running.append(loss.detach().item())

            val_loss = self.evaluate(val_loader, use_ema=True)
            tr_mean = float(np.mean(running)) if running else float('nan')
            print(f"[{model_name}] [Epoch {epoch:03d}] train_loss: {tr_mean:.5f}  val_loss: {val_loss:.5f}")

            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch} (patience={patience})")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state, strict=True)
        return self.model, best_val

# =====================
# Build helper (TCN만)
# =====================
def build_tcn(cfg: EnhancedNHiTSConfig, ds: EnhancedNHiTSDataset, C=128, depth=4, drop=0.1):
    cal_dim = int(ds.cal_feats.shape[1] + getattr(ds, "extra_dim", 0))
    return TCNTiny(cfg, cfg.in_len, cfg.out_len, cal_dim, ds.n_stores, ds.n_categories, ds.n_types, C=C, depth=depth, drop=drop)

# =====================
# Main
# =====================
if __name__ == "__main__":
    cfg = EnhancedNHiTSConfig()
    if cfg.store_weights is None: cfg.store_weights = DEFAULT_STORE_WEIGHTS
    if cfg.custom_holidays_list is None: cfg.custom_holidays_list = DEFAULT_CUSTOM_HOLIDAYS
    set_seed(cfg.seed)

    if not os.path.exists(cfg.train_csv):
        raise FileNotFoundError(f"학습 파일을 찾을 수 없습니다: {cfg.train_csv}")

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    # Load train
    train_df = pd.read_csv(cfg.train_csv)
    ds = EnhancedNHiTSDataset(cfg, train_df)

    # 정보 출력
    print(f"Base cal_dim = {ds.cal_feats.shape[1]}, extra_dim = {ds.extra_dim}, USE_CORR={cfg.USE_CORR_FEATURES}, USE_DOW={cfg.USE_DOW_STRENGTH}")

    # Validation: 마지막 주(예시)로 마스크
    cv_end = "2024-06-14"
    mask_val = make_val_mask_by_week(ds, cv_end)

    # Build TCN
    model = build_tcn(cfg, ds, C=128, depth=4, drop=0.1)

    # Train
    trainer = GenericTrainer(cfg, ds, model,
                             epochs=cfg.EPOCHS_FULL, batch_size=cfg.BATCH_FULL,
                             base_lr=cfg.BASE_LR_FULL, max_lr=cfg.MAX_LR_FULL, weight_decay=cfg.WD_FULL)
    train_loader, val_loader = trainer.make_loaders_from_mask(mask_val)
    model, best_val = trainer.train_with_loaders(train_loader, val_loader, model_name="TCN")

    # Save checkpoint
    ckpt_path = os.path.join(cfg.checkpoint_dir, "TCN_best_fold.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Saved best checkpoint to {ckpt_path} (val_loss={best_val:.6f})")


Base cal_dim = 32, extra_dim = 1, USE_CORR=False, USE_DOW=True
[TCN] [Epoch 001] train_loss: 0.41031  val_loss: 0.86735
[TCN] [Epoch 002] train_loss: 0.36088  val_loss: 0.79279
[TCN] [Epoch 003] train_loss: 0.34895  val_loss: 0.74820
[TCN] [Epoch 004] train_loss: 0.34219  val_loss: 0.72994
[TCN] [Epoch 005] train_loss: 0.33679  val_loss: 0.73105
[TCN] [Epoch 006] train_loss: 0.33065  val_loss: 0.73783
[TCN] [Epoch 007] train_loss: 0.32659  val_loss: 0.74188
[TCN] [Epoch 008] train_loss: 0.32108  val_loss: 0.73853
[TCN] [Epoch 009] train_loss: 0.31705  val_loss: 0.72735
[TCN] [Epoch 010] train_loss: 0.31348  val_loss: 0.70849
[TCN] [Epoch 011] train_loss: 0.30963  val_loss: 0.68321
[TCN] [Epoch 012] train_loss: 0.30675  val_loss: 0.65503
[TCN] [Epoch 013] train_loss: 0.30315  val_loss: 0.62706
[TCN] [Epoch 014] train_loss: 0.30048  val_loss: 0.60055
[TCN] [Epoch 015] train_loss: 0.29811  val_loss: 0.57679
[TCN] [Epoch 016] train_loss: 0.29529  val_loss: 0.55464
[TCN] [Epoch 017] train_l

Exception in thread Thread-208 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource

KeyboardInterrupt: 

# 둘다 도입
[TCN] [Epoch 050] train_loss: 0.25660  val_loss: 0.37780


In [5]:
# tcn_only_with_corr_and_dow.py
# -*- coding: utf-8 -*-
import os, gc, math, random, warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# =====================
# Config
# =====================
@dataclass
class EnhancedNHiTSConfig:
    # Paths
    train_csv: str = "/content/drive/MyDrive/data/train/train_original.csv"
    test_glob: str = "/content/drive/MyDrive/data/test/*.csv"
    submission_template_csv: str = "/content/drive/MyDrive/data/sample_submission.csv"
    out_submission_csv: str = "/content/drive/MyDrive/data/0823_submission.csv"
    checkpoint_dir: str = "/content/drive/MyDrive/data/checkpoint/enhanced_checkpoints"

    # Columns
    date_col: str = "영업일자"
    item_col: str = "영업장명_메뉴명"
    target_col: str = "매출수량"

    # Window
    in_len: int = 28
    out_len: int = 7

    # Train filtering
    train_end_date: str = "2024-06-15"

    # General
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log1p: bool = True

    # Dataloaders
    num_workers: int = 4
    pin_memory: bool = True
    persistent_workers: bool = False

    # Loss
    eps_smape: float = 0.01
    zero_weight: float = 0.01
    hurdle_lambda: float = 0.15

    # AMP/EMA
    use_amp: bool = True
    use_compile: bool = False
    ema_decay: float = 0.9998

    # Train hyperparams
    EPOCHS_FULL: int = 50
    BATCH_FULL: int = 256
    BASE_LR_FULL: float = 6e-4
    MAX_LR_FULL: float = 1.5e-3
    WD_FULL: float = 6e-4

    # Early stop / grad clip
    grad_clip: float = 0.5
    earlystop_patience_ratio: float = 0.12
    earlystop_patience_min: int = 8

    # Optional weights/holidays (채워지지 않으면 기본 사용)
    store_weights: Optional[Dict[str, float]] = None
    custom_holidays_list: Optional[List[str]] = None

    # ====== NEW: Feature toggles & params ======
    # Corr-based features
    USE_CORR_FEATURES: bool = True
    CORR_THRESHOLD: float = 0.5     # 상관 임계치
    CORR_TOPN: int = 5              # 메뉴당 상위 파트너 N개 제한 (0 = 제한 없음)
    CORR_LAGS: Tuple[int, ...] = (1, 7)
    CORR_RMEANS: Tuple[int, ...] = (7, 14)  # 모두 shift=1

    # Day-of-week strength
    USE_DOW_STRENGTH: bool = True
    DOW_RATIO_THRESHOLD: float = 2.0    # 평균 대비 2배 이상
    DOW_MIN_SUPPORT: int = 4            # 요일 평균 계산에 필요한 최소 관측일 수

    # Scaling extra features: "none" | "log1p" | "zscore"
    EXTRA_FEAT_SCALING: str = "none"


DEFAULT_STORE_WEIGHTS = {
    "미라시아": 1, "담하": 1, "연회장": 1, "라그로타": 1,
    "느티나무 셀프BBQ": 1, "화담숲주막": 1, "카페테리아": 1,
    "화담숲카페": 1, "포레스트릿": 1,
}
DEFAULT_CUSTOM_HOLIDAYS = [
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-01',
    '2023-05-05','2023-05-27','2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30',
    '2023-10-02','2023-10-03','2023-10-09','2023-12-25',
    '2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01','2024-04-10',
    '2024-05-01','2024-05-05','2024-05-06','2024-05-15','2024-06-06','2024-08-15','2024-09-16',
    '2024-09-17','2024-09-18','2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    '2025-01-01','2025-01-28','2025-01-29','2025-01-30','2025-03-01','2025-03-03','2025-05-01',
    '2025-05-05','2025-05-06','2025-06-06','2025-08-15'
]

# =====================
# Utils
# =====================
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sine_cosine_encoding(value: float, max_val: float):
    return math.sin(2 * math.pi * value / max_val), math.cos(2 * math.pi * value / max_val)

def parse_store_name(item_full: str) -> str:
    return item_full.split("_")[0]

def parse_menu_name(item_full: str) -> str:
    return item_full.split("_", 1)[1] if "_" in item_full else item_full

def build_enhanced_features(dates: List[pd.Timestamp], holidays_set: set,
                            store_names: Optional[List[str]] = None) -> pd.DataFrame:
    df = pd.DataFrame({"date": dates})
    df["dow"] = df["date"].dt.weekday
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["week"] = df["date"].apply(lambda d: int(d.isocalendar().week))
    df["quarter"] = df["date"].dt.quarter
    df["is_friday"] = (df["dow"] == 4).astype(int)
    df["is_saturday"] = (df["dow"] == 5).astype(int)
    df["is_sunday"] = (df["dow"] == 6).astype(int)
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["is_holiday"] = df["date"].isin(holidays_set).astype(int)
    df["is_off"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)
    df["tomorrow"] = df["date"] + pd.Timedelta(days=1)
    df["yesterday"] = df["date"] - pd.Timedelta(days=1)
    df["is_before_holiday"] = df["tomorrow"].isin(holidays_set).astype(int)
    df["is_after_holiday"] = df["yesterday"].isin(holidays_set).astype(int)
    df["is_month_start"] = (df["day"] <= 3).astype(int)
    df["is_month_end"] = (df["day"] >= 28).astype(int)
    df["is_spring"] = df["month"].isin([4, 5, 6]).astype(int)
    df["is_summer"] = df["month"].isin([7, 8]).astype(int)
    df["is_autumn"] = df["month"].isin([9, 10, 11]).astype(int)
    df["is_winter"] = df["month"].isin([12, 1, 2, 3]).astype(int)
    df["is_summer_vacation"] = df["month"].isin([7, 8]).astype(int)
    df["is_winter_vacation"] = df["month"].isin([12, 1, 2]).astype(int)

    # 날짜 기반 정기 휴무일 (원본 로직 요약)
    df['is_regular_holiday'] = 0
    df.loc[((df['date'].dt.month.isin(range(2, 12))) & (df['date'].dt.year == 2023) & (df['dow'] == 0)) |
           ((df['date'].dt.month.isin(range(3, 7))) & (df['date'].dt.year == 2024) & (df['dow'] == 0)) |
           (df['date'].isin(['2023-03-01'])) |
           (df['date'].isin(['2023-09-01','2023-09-02','2023-09-03']) & (df['dow'].isin([0,1,2]))) |
           (df['date'].isin(['2024-03-01'])) |
           ((df['date'] >= '2023-05-01') & (df['dow'].isin([0,1,2,3]))) |
           (df['date'].dt.month.isin([12,1,2,3]) & df['date'].dt.year.isin([2023,2024,2025])) |
           ((~df['date'].dt.month.isin([12,1,2,3])) & (df['dow'] == 0)), 'is_regular_holiday'] = 1

    df[["month_sin", "month_cos"]] = df["month"].apply(lambda m: pd.Series(sine_cosine_encoding(m, 12)))
    df[["dow_sin", "dow_cos"]] = df["dow"].apply(lambda d: pd.Series(sine_cosine_encoding(d, 7)))
    df[["doy_sin", "doy_cos"]] = df["date"].apply(lambda d: pd.Series(sine_cosine_encoding(d.dayofyear, 365)))
    df[["week_sin", "week_cos"]] = df["week"].apply(lambda w: pd.Series(sine_cosine_encoding(w, 52)))
    df[["quarter_sin", "quarter_cos"]] = df["quarter"].apply(lambda q: pd.Series(sine_cosine_encoding(q, 4)))
    return df.drop(columns=["tomorrow", "yesterday"])

def get_menu_category(menu_name: str) -> str:
    menu_lower = menu_name.lower()
    if any(word in menu_lower for word in ['막걸리','소주','맥주','와인','참이슬','처음처럼','카스','하이네켄','버드와이저','스텔라']):
        return 'alcohol'
    elif any(word in menu_lower for word in ['찌개','탕','국밥','라면','해장국','갈비탕']):
        return 'hot_food'
    elif any(word in menu_lower for word in ['삼겹','갈비','목살','bbq','구이','불고기']):
        return 'bbq'
    elif any(word in menu_lower for word in ['아이스크림','식혜','콜라','스프라이트','에이드']):
        return 'dessert_drink'
    elif any(word in menu_lower for word in ['아메리카노','라떼','커피']):
        return 'coffee'
    elif any(word in menu_lower for word in ['냉면','파스타','스파게티','면','우동']):
        return 'noodles'
    elif any(word in menu_lower for word in ['비빔밥','볶음밥','공깃밥','정식']):
        return 'rice'
    else:
        return 'others'

def get_store_type(store_name: str) -> str:
    if store_name in ["느티나무 셀프BBQ", "연회장"]:
        return 'Special_Occasion'
    elif store_name in ["화담숲주막", "화담숲카페"]:
        return 'Forest'
    elif store_name in ["담하", "미라시아"]:
        return 'Fine_Dining'
    elif store_name in ["카페테리아", "포레스트릿"]:
        return 'Casual'
    else:
        return 'Unique_Venue'

def get_weekend_sales_ratio(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy['영업일자'] = pd.to_datetime(df_copy['영업일자'])
    df_copy['요일'] = df_copy['영업일자'].dt.weekday
    weekend_df = df_copy[df_copy['요일'].isin([5,6])]
    weekday_df = df_copy[~df_copy['요일'].isin([5,6])]
    weekend_sales = weekend_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    weekday_sales = weekday_df.groupby('영업장명_메뉴명')['매출수량'].mean()
    sales_ratio = (weekend_sales / weekday_sales).fillna(1.0).reset_index()
    sales_ratio.rename(columns={'매출수량':'weekend_sales_ratio'}, inplace=True)
    sales_ratio['weekend_sales_ratio'] = sales_ratio['weekend_sales_ratio'].replace([float('inf'), -float('inf')], 1.0)
    return sales_ratio

# ===== Correlation-based & DOW-strong features =====
def _safe_shift(a: pd.Series, n: int) -> pd.Series:
    return a.shift(n)

def _safe_rmean(a: pd.Series, w: int) -> pd.Series:
    return a.shift(1).rolling(window=w, min_periods=1).mean()

def build_corr_feature_bank(
    pivot: pd.DataFrame,              # index=date, columns=item, values=sales
    cutoff_date: pd.Timestamp,        # train_end_date (누수 방지)
    threshold: float = 0.5,
    topn: int = 5,
    lags: Tuple[int, ...] = (1, 7),
    rmeans: Tuple[int, ...] = (7, 14),
) -> Tuple[Dict[str, List[str]], Dict[str, pd.DataFrame]]:
    # train 구간만으로 상관 계산
    pivot_train = pivot.loc[:cutoff_date]
    corr = pivot_train.corr(method="pearson").fillna(0.0)

    corr_map: Dict[str, List[str]] = {}
    for tgt in corr.columns:
        partners_all = corr.index[(corr[tgt] >= threshold) & (corr.index != tgt)].tolist()
        # 상관값 기준 정렬 후 상위 N 제한
        partners_all = sorted(partners_all, key=lambda it: corr.loc[it, tgt], reverse=True)
        if topn and topn > 0:
            partners_all = partners_all[:topn]
        corr_map[tgt] = partners_all

    feat_bank: Dict[str, pd.DataFrame] = {}
    for tgt in pivot.columns:
        df_list = []
        for partner in corr_map.get(tgt, []):
            s = pivot[partner].astype(float)
            for L in lags:
                df_list.append(_safe_shift(s, L).rename(f"{partner}_lag{L}"))
            for W in rmeans:
                df_list.append(_safe_rmean(s, W).rename(f"{partner}_rmean{W}_lag1"))
        if df_list:
            fb = pd.concat(df_list, axis=1)
        else:
            fb = pd.DataFrame(index=pivot.index)
        feat_bank[tgt] = fb
    return corr_map, feat_bank

def compute_dow_strength_flags(df: pd.DataFrame,
                               date_col: str, item_col: str, target_col: str,
                               cutoff_date: pd.Timestamp,
                               ratio_threshold: float = 2.0,
                               min_support: int = 4) -> pd.DataFrame:
    tmp = df.copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.loc[tmp[date_col] <= cutoff_date]  # 누수 방지
    tmp["dow"] = tmp[date_col].dt.weekday

    overall = tmp.groupby(item_col)[target_col].mean()
    by_dow = tmp.groupby([item_col, "dow"])[target_col].agg(['mean','count']).unstack("dow")
    # 구조: columns MultiIndex [('mean',0..6), ('count',0..6)]
    mean_mat = by_dow['mean'].fillna(0.0)
    cnt_mat  = by_dow['count'].fillna(0.0)

    ratio = mean_mat.divide(overall, axis=0).fillna(0.0)
    strong = (ratio >= ratio_threshold) & (cnt_mat >= min_support)

    # 날짜 index × item 행렬 플래그 생성
    all_dates = pd.date_range(tmp[date_col].min(), df[date_col].max())
    items = tmp[item_col].unique().tolist()
    flag = pd.DataFrame(0, index=all_dates, columns=items, dtype=np.int8)
    for it in items:
        strong_dows = set(np.where(strong.loc[it].values)[0]) if it in strong.index else set()
        if not strong_dows:
            continue
        # 날짜별 dow가 strong이면 1
        dows = pd.Series(all_dates.weekday, index=all_dates)
        flag.loc[dows.isin(strong_dows), it] = 1
    return flag

# =====================
# Dataset
# =====================
class EnhancedNHiTSDataset(Dataset):
    """
    원본의 Dense 보강/캘린더피처/주말비율/인덱싱 로직 + (옵션) 상관 기반 lag/rmean, 요일강세 플래그
    """
    def __init__(self, cfg: EnhancedNHiTSConfig, df: pd.DataFrame):
        self.cfg = cfg
        self.df = df.copy()
        if cfg.date_col not in self.df.columns or cfg.item_col not in self.df.columns or cfg.target_col not in self.df.columns:
            raise ValueError("입력 데이터에 필요한 컬럼이 없습니다.")
        self.df[cfg.date_col] = pd.to_datetime(self.df[cfg.date_col])
        self.df[cfg.target_col] = pd.to_numeric(self.df[cfg.target_col], errors='coerce').fillna(0.0).clip(lower=0)

        # Dense dataset 보강 (모든 날짜×아이템)
        all_dates = pd.date_range(start=self.df[cfg.date_col].min(), end=self.df[cfg.date_col].max())
        all_items = self.df[cfg.item_col].unique()
        full_df = pd.MultiIndex.from_product([all_dates, all_items], names=[cfg.date_col, cfg.item_col]).to_frame(index=False)
        self.df = pd.merge(full_df, self.df, on=[cfg.date_col, cfg.item_col], how='left')
        self.df[cfg.target_col] = self.df[cfg.target_col].fillna(0)

        # 주말 상대 판매량 비율 병합
        weekend_ratio_df = get_weekend_sales_ratio(self.df)
        self.df = pd.merge(self.df, weekend_ratio_df, on=cfg.item_col, how='left')
        self.df['weekend_sales_ratio'].fillna(1.0, inplace=True)

        # 기본 피벗
        pivot = self.df.pivot(index=cfg.date_col, columns=cfg.item_col, values=cfg.target_col).sort_index()
        self.items = list(pivot.columns)
        self.dates = list(pivot.index)
        self.values = pivot.fillna(0.0).values.astype(np.float32)

        # 캘린더 피처
        holidays_set = set(pd.to_datetime(cfg.custom_holidays_list)) if cfg.custom_holidays_list else set()
        caldf = build_enhanced_features(self.dates, holidays_set)
        self.cal_feats = caldf.drop(columns=["date"]).values.astype(np.float32)
        self.cal_feat_names = [c for c in caldf.columns if c != "date"]

        # 인덱싱용 메타
        stores = [parse_store_name(it) for it in self.items]
        menus  = [parse_menu_name(it) for it in self.items]

        self.store2idx = {s: i for i, s in enumerate(sorted(set(stores)))}
        self.item_store_idx = np.array([self.store2idx[s] for s in stores], dtype=np.int64)
        self.n_stores = len(self.store2idx)

        self.cat2idx = {c: i for i, c in enumerate(sorted(set([get_menu_category(m) for m in menus])))}
        self.item_cat_idx = np.array([self.cat2idx[get_menu_category(m)] for m in menus], dtype=np.int64)
        self.n_categories = len(self.cat2idx)

        self.type2idx = {t: i for i, t in enumerate(sorted(set([get_store_type(s) for s in stores])))}
        self.item_type_idx = np.array([self.type2idx[get_store_type(s)] for s in stores], dtype=np.int64)
        self.n_types = len(self.type2idx)

        # ===== Extra features (옵션) =====
        cutoff = pd.to_datetime(cfg.train_end_date)
        self.extra_dim = 0
        self.extra_feats_per_item: List[np.ndarray] = []

        # (A) 요일 강세 플래그
        if cfg.USE_DOW_STRENGTH:
            dow_flag = compute_dow_strength_flags(
                self.df, cfg.date_col, cfg.item_col, cfg.target_col,
                cutoff_date=cutoff, ratio_threshold=cfg.DOW_RATIO_THRESHOLD, min_support=cfg.DOW_MIN_SUPPORT
            )
            dow_flag = dow_flag.reindex(index=pivot.index, columns=pivot.columns).fillna(0).astype(np.float32)
        else:
            dow_flag = pd.DataFrame(index=pivot.index, columns=pivot.columns, data=0.0, dtype=np.float32)

        # (B) 상관 기반 lag/rolling
        if cfg.USE_CORR_FEATURES:
            corr_map, feat_bank = build_corr_feature_bank(
                pivot=pivot, cutoff_date=cutoff,
                threshold=cfg.CORR_THRESHOLD, topn=cfg.CORR_TOPN,
                lags=cfg.CORR_LAGS, rmeans=cfg.CORR_RMEANS
            )
        else:
            corr_map, feat_bank = {}, {it: pd.DataFrame(index=pivot.index) for it in self.items}

        # 아이템별 extra DataFrame 구성 + 요일 강세 1채널 추가
        for it in self.items:
            fb = feat_bank.get(it, pd.DataFrame(index=pivot.index))
            fb = fb.copy()
            fb["_dow_strong"] = dow_flag[it] if it in dow_flag.columns else 0.0
            fb = fb.reindex(index=pivot.index).fillna(0.0).astype(np.float32)

            # 스케일링 옵션
            if fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "log1p":
                fb = np.log1p(fb)
            elif fb.shape[1] and self.cfg.EXTRA_FEAT_SCALING.lower() == "zscore":
                mu = fb.mean(axis=0).replace(0.0, 0.0)
                std = fb.std(axis=0).replace(0.0, 1.0)
                fb = (fb - mu) / std

            self.extra_feats_per_item.append(fb.values)

        if self.extra_feats_per_item and self.extra_feats_per_item[0].shape[1] > 0:
            # 열 수를 모든 아이템에서 동일하게 맞춤
            min_dim = min(arr.shape[1] for arr in self.extra_feats_per_item)
            self.extra_feats_per_item = [arr[:, :min_dim] for arr in self.extra_feats_per_item]
            self.extra_dim = min_dim
        else:
            self.extra_dim = 0

        # 샘플 가중치
        sw = self.cfg.store_weights or {}
        self.sample_weights = np.array([sw.get(parse_store_name(it), 1.0) for it in self.items], dtype=np.float32)

        # 인덱스 생성
        self.indices: List[Tuple[int,int]] = []
        self.target_end_dates: List[pd.Timestamp] = []
        T = len(self.dates)
        Lx, Ly = cfg.in_len, cfg.out_len
        cutoff_date = pd.to_datetime(cfg.train_end_date)
        max_start = max(0, T - (Lx + Ly))
        for j in range(len(self.items)):
            for t0 in range(0, max_start + 1):
                end_date = self.dates[t0 + Lx + Ly - 1]
                if end_date <= cutoff_date:
                    self.indices.append((t0, j))
                    self.target_end_dates.append(end_date)

        self.target_end_dates = np.asarray(self.target_end_dates, dtype='datetime64[ns]')

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        cfg = self.cfg
        t0, j = self.indices[idx]
        Lx, Ly = cfg.in_len, cfg.out_len

        x = self.values[t0:t0 + Lx, j]
        y = self.values[t0 + Lx:t0 + Lx + Ly, j]

        if cfg.log1p:
            x_in = np.log1p(x); y_out = np.log1p(y)
        else:
            x_in, y_out = x.copy(), y.copy()

        past_cal = self.cal_feats[t0:t0 + Lx, :]
        fut_cal  = self.cal_feats[t0 + Lx:t0 + Lx + Ly, :]

        # 추가 피처
        if self.extra_dim > 0:
            ex_full = self.extra_feats_per_item[j]  # [T, E]
            past_ex = ex_full[t0:t0 + Lx, :]
            fut_ex  = np.zeros((Ly, self.extra_dim), dtype=np.float32)  # 미래는 0
        else:
            past_ex = np.zeros((Lx, 0), dtype=np.float32)
            fut_ex  = np.zeros((Ly, 0), dtype=np.float32)

        store_idx = self.item_store_idx[j]
        cat_idx   = self.item_cat_idx[j]
        type_idx  = self.item_type_idx[j]
        sample_w  = self.sample_weights[j]
        pos_mask  = (y > 0).astype(np.float32)

        zero_ratio = np.mean(x == 0)
        weekend_factor = np.mean(past_cal[:, 12])  # (참고) 예전 코드와 동일한 위치 피처 예시

        return {
            "x": torch.from_numpy(x_in).float(),
            "y": torch.from_numpy(y_out).float(),
            "past_cal": torch.from_numpy(past_cal).float(),
            "fut_cal": torch.from_numpy(fut_cal).float(),
            "past_ex": torch.from_numpy(past_ex).float(),
            "fut_ex": torch.from_numpy(fut_ex).float(),
            "store_idx": torch.tensor(store_idx, dtype=torch.long),
            "cat_idx": torch.tensor(cat_idx, dtype=torch.long),
            "type_idx": torch.tensor(type_idx, dtype=torch.long),
            "sample_w": torch.tensor(sample_w, dtype=torch.float32),
            "pos_mask": torch.from_numpy(pos_mask).float(),
            "zero_ratio": torch.tensor(zero_ratio, dtype=torch.float32),
            "weekend_factor": torch.tensor(weekend_factor, dtype=torch.float32),
            # 주말 비율은 필요 시 추가 가능
        }

def make_val_mask_by_week(dataset: 'EnhancedNHiTSDataset', end_date_str: str) -> np.ndarray:
    end_date = pd.to_datetime(end_date_str)
    start_date = end_date - pd.Timedelta(days=6)
    ted = dataset.target_end_dates
    return (ted >= start_date) & (ted <= end_date)

# =====================
# TCN (sequence head + meta)
# =====================
class _TCNBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3, d=1, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c_in, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(), nn.Dropout(p),
            nn.Conv1d(c_out, c_out, kernel_size=k, dilation=d, padding='same'), nn.GELU(),
        )
        self.proj = nn.Conv1d(c_in, c_out, kernel_size=1) if c_in!=c_out else nn.Identity()
        self.norm = nn.LayerNorm(c_out)

    def forward(self, x):
        y = self.net(x) + self.proj(x)
        return self.norm(y.transpose(1,2)).transpose(1,2)

class TCNTiny(nn.Module):
    def __init__(self, cfg, in_len, out_len, cal_dim, n_stores, n_categories, n_types, C=128, depth=4, drop=0.1):
        super().__init__()
        self.out_len = out_len
        self.stem = nn.Conv1d(1, C, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([_TCNBlock(C, C, k=3, d=2**i, p=drop) for i in range(depth)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(C, out_len))

        self.store_emb = nn.Embedding(n_stores,64)
        self.cat_emb   = nn.Embedding(n_categories,32)
        self.type_emb  = nn.Embedding(n_types,16)
        self.cal_proj  = nn.Linear(cal_dim,128)

        self.meta_head = nn.Sequential(nn.Linear(64+32+16+128, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))
        self.prob_head = nn.Sequential(nn.Linear(64+32+16+128+1, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, out_len))

    def forward(self, x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                weekend_ratio=None, past_ex=None, fut_ex=None):
        # sequence path
        z = self.stem(x.unsqueeze(1))
        for blk in self.blocks:
            z = blk(z)
        seq_out = self.head(z)

        # extra concat
        if (past_ex is not None) and (fut_ex is not None) and (past_ex.numel() > 0):
            past_cal = torch.cat([past_cal, past_ex], dim=-1)
            fut_cal  = torch.cat([fut_cal,  fut_ex ], dim=-1)

        # meta path
        store = self.store_emb(store_idx); cat = self.cat_emb(cat_idx); typ = self.type_emb(type_idx)
        cal_all = torch.cat([past_cal, fut_cal], dim=1).mean(dim=1)  # [B, cal+extra]
        cal = self.cal_proj(cal_all)
        meta = torch.cat([store, cat, typ, cal], dim=-1)

        value_pred  = seq_out + self.meta_head(meta)
        prob_logits = self.prob_head(torch.cat([meta, x.mean(dim=1, keepdim=True)], dim=-1))
        return value_pred, prob_logits

# =====================
# Loss / EMA
# =====================
class UltraEnhancedHurdleLoss(nn.Module):
    def __init__(self, eps: float = 0.01, zero_weight: float = 0.01, lambda_bce: float = 0.15):
        super().__init__()
        self.eps = eps; self.zero_weight = zero_weight; self.lambda_bce = lambda_bce
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, v_pred_log, p_logits, y_true_log, pos_mask, sample_w):
        z = (pos_mask > 0).float()
        bce = self.bce(p_logits, z)

        yp_val = torch.expm1(v_pred_log).clamp_min(0.0)
        yt_val = torch.expm1(y_true_log).clamp_min(0.0)

        ultra_eps = torch.where(yt_val < 0.5, self.eps * 0.2,
                                torch.where(yt_val < 2.0, self.eps * 0.5, self.eps))
        denom = (torch.abs(yp_val) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_pos = 2.0 * torch.abs(yp_val - yt_val) / denom
        smape_pos = smape_pos * (yt_val > 0).float()

        p = torch.sigmoid(p_logits)
        y_hat = p * yp_val
        denom2 = (torch.abs(y_hat) + torch.abs(yt_val)).clamp_min(ultra_eps)
        smape_all = 2.0 * torch.abs(y_hat - yt_val) / denom2

        ultra_zero_weight = torch.where(
            yt_val < 0.01, torch.full_like(yt_val, self.zero_weight * 0.1),
            torch.where(yt_val < 0.1, torch.full_like(yt_val, self.zero_weight * 0.3),
                        torch.where(yt_val < 1.0, torch.full_like(yt_val, self.zero_weight * 0.6), torch.ones_like(yt_val)))
        )
        smape_all = smape_all * ultra_zero_weight

        bce_s = bce.mean(dim=1); pos_s = smape_pos.mean(dim=1); all_s = smape_all.mean(dim=1)
        sample_loss = self.lambda_bce * bce_s + 0.4 * pos_s + 0.6 * all_s

        sw = sample_w.view(-1); wsum = sw.sum().clamp_min(1e-8)
        loss = (sample_loss * sw).sum() / wsum
        return loss, sample_loss.detach(), sw.detach()

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.shadow = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

    def update(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    self.shadow[name].mul_((self.decay)).add_(p.detach(), alpha=1-self.decay)

    def apply_to(self, model: nn.Module):
        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    p.copy_(self.shadow[name])

# =====================
# Trainer
# =====================
class GenericTrainer:
    def __init__(self, cfg: EnhancedNHiTSConfig, dataset: EnhancedNHiTSDataset, model: nn.Module,
                 epochs: int, batch_size: int, base_lr: float, max_lr: float, weight_decay: float):
        self.cfg = cfg; self.dataset = dataset; self.model = model.to(cfg.device)
        self.device = torch.device(cfg.device)
        self.criterion = UltraEnhancedHurdleLoss(cfg.eps_smape, cfg.zero_weight, cfg.hurdle_lambda)
        self.ema = EMA(self.model, decay=cfg.ema_decay)
        self.epochs = epochs; self.batch_size = batch_size
        self.base_lr = base_lr; self.max_lr = max_lr; self.weight_decay = weight_decay

    def make_loaders_from_mask(self, mask_val: np.ndarray):
        idx_all = np.arange(len(self.dataset))
        val_idx = idx_all[mask_val]; train_idx = idx_all[~mask_val]
        train_subset = torch.utils.data.Subset(self.dataset, train_idx)
        val_subset = torch.utils.data.Subset(self.dataset, val_idx)
        train_loader = DataLoader(
            train_subset, batch_size=self.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        val_loader = DataLoader(
            val_subset, batch_size=self.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory,
            persistent_workers=self.cfg.persistent_workers, drop_last=False
        )
        return train_loader, val_loader

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, use_ema: bool = True) -> float:
        if loader is None: return float('inf')
        self.model.eval()
        backup = None
        if use_ema:
            backup = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            self.ema.apply_to(self.model)

        total_loss, total_weight = 0.0, 0.0
        amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
            self.cfg.use_amp and torch.cuda.is_available() and self.device.type == "cuda"
        ) else torch.cuda.amp.autocast(enabled=False)

        with amp_ctx:
            for batch in loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                v_pred, p_logits = self.model(
                    x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                    weekend_ratio=None,
                    past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                    fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                )
                _, sample_loss, sw = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                total_loss += (sample_loss * sw).sum().item()
                total_weight += sw.sum().item()

        if use_ema and backup is not None:
            self.model.load_state_dict(backup); del backup
            gc.collect();
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        if total_weight <= 0: return float('inf')
        return total_loss / total_weight

    def train_with_loaders(self, train_loader: DataLoader, val_loader: DataLoader, model_name: str = "TCN"):
        optim = torch.optim.AdamW(self.model.parameters(), lr=self.base_lr, weight_decay=self.weight_decay, betas=(0.9, 0.999))
        steps_per_epoch = max(1, len(train_loader))
        sched = torch.optim.lr_scheduler.OneCycleLR(
            optim, max_lr=self.max_lr, epochs=self.epochs,
            steps_per_epoch=steps_per_epoch, pct_start=0.05,
            div_factor=max(1e-8, self.max_lr / max(1e-8, self.base_lr))
        )
        patience = max(self.cfg.earlystop_patience_min, int(self.epochs * self.cfg.earlystop_patience_ratio))
        best_val = float("inf"); best_state = None; no_improve = 0

        for epoch in range(1, self.epochs+1):
            self.model.train()
            amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if (
                self.cfg.use_amp and torch.cuda.is_available() and self.device.type=="cuda"
            ) else torch.cuda.amp.autocast(enabled=False)
            running = []
            for batch in train_loader:
                x = batch["x"].to(self.device); y = batch["y"].to(self.device)
                past_cal = batch["past_cal"].to(self.device); fut_cal = batch["fut_cal"].to(self.device)
                store_idx = batch["store_idx"].to(self.device); cat_idx = batch["cat_idx"].to(self.device)
                type_idx = batch["type_idx"].to(self.device); sample_w = batch["sample_w"].to(self.device)
                pos_mask = batch["pos_mask"].to(self.device)

                with amp_ctx:
                    v_pred, p_logits = self.model(
                        x, past_cal, fut_cal, store_idx, cat_idx, type_idx,
                        weekend_ratio=None,
                        past_ex=batch.get("past_ex", None).to(self.device) if "past_ex" in batch else None,
                        fut_ex=batch.get("fut_ex", None).to(self.device) if "fut_ex" in batch else None,
                    )
                    loss, _, _ = self.criterion(v_pred, p_logits, y, pos_mask, sample_w)

                optim.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                optim.step(); sched.step(); self.ema.update(self.model)
                running.append(loss.detach().item())

            val_loss = self.evaluate(val_loader, use_ema=True)
            tr_mean = float(np.mean(running)) if running else float('nan')
            print(f"[{model_name}] [Epoch {epoch:03d}] train_loss: {tr_mean:.5f}  val_loss: {val_loss:.5f}")

            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch} (patience={patience})")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state, strict=True)
        return self.model, best_val

# =====================
# Build helper (TCN만)
# =====================
def build_tcn(cfg: EnhancedNHiTSConfig, ds: EnhancedNHiTSDataset, C=128, depth=4, drop=0.1):
    cal_dim = int(ds.cal_feats.shape[1] + getattr(ds, "extra_dim", 0))
    return TCNTiny(cfg, cfg.in_len, cfg.out_len, cal_dim, ds.n_stores, ds.n_categories, ds.n_types, C=C, depth=depth, drop=drop)

# =====================
# Main
# =====================
if __name__ == "__main__":
    cfg = EnhancedNHiTSConfig()
    if cfg.store_weights is None: cfg.store_weights = DEFAULT_STORE_WEIGHTS
    if cfg.custom_holidays_list is None: cfg.custom_holidays_list = DEFAULT_CUSTOM_HOLIDAYS
    set_seed(cfg.seed)

    if not os.path.exists(cfg.train_csv):
        raise FileNotFoundError(f"학습 파일을 찾을 수 없습니다: {cfg.train_csv}")

    os.makedirs(cfg.checkpoint_dir, exist_ok=True)

    # Load train
    train_df = pd.read_csv(cfg.train_csv)
    ds = EnhancedNHiTSDataset(cfg, train_df)

    # 정보 출력
    print(f"Base cal_dim = {ds.cal_feats.shape[1]}, extra_dim = {ds.extra_dim}, USE_CORR={cfg.USE_CORR_FEATURES}, USE_DOW={cfg.USE_DOW_STRENGTH}")

    # Validation: 마지막 주(예시)로 마스크
    cv_end = "2024-06-14"
    mask_val = make_val_mask_by_week(ds, cv_end)

    # Build TCN
    model = build_tcn(cfg, ds, C=128, depth=4, drop=0.1)

    # Train
    trainer = GenericTrainer(cfg, ds, model,
                             epochs=cfg.EPOCHS_FULL, batch_size=cfg.BATCH_FULL,
                             base_lr=cfg.BASE_LR_FULL, max_lr=cfg.MAX_LR_FULL, weight_decay=cfg.WD_FULL)
    train_loader, val_loader = trainer.make_loaders_from_mask(mask_val)
    model, best_val = trainer.train_with_loaders(train_loader, val_loader, model_name="TCN")

    # Save checkpoint
    ckpt_path = os.path.join(cfg.checkpoint_dir, "TCN_best_fold.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Saved best checkpoint to {ckpt_path} (val_loss={best_val:.6f})")


Base cal_dim = 32, extra_dim = 1, USE_CORR=True, USE_DOW=True
[TCN] [Epoch 001] train_loss: 0.40741  val_loss: 0.86842
[TCN] [Epoch 002] train_loss: 0.35944  val_loss: 0.79237
[TCN] [Epoch 003] train_loss: 0.34919  val_loss: 0.74238
[TCN] [Epoch 004] train_loss: 0.34102  val_loss: 0.72049
[TCN] [Epoch 005] train_loss: 0.33549  val_loss: 0.71790
[TCN] [Epoch 006] train_loss: 0.32933  val_loss: 0.72432
[TCN] [Epoch 007] train_loss: 0.32519  val_loss: 0.72945
[TCN] [Epoch 008] train_loss: 0.32007  val_loss: 0.72729
[TCN] [Epoch 009] train_loss: 0.31593  val_loss: 0.71592
[TCN] [Epoch 010] train_loss: 0.31267  val_loss: 0.69551
[TCN] [Epoch 011] train_loss: 0.30894  val_loss: 0.66939
[TCN] [Epoch 012] train_loss: 0.30560  val_loss: 0.64156
[TCN] [Epoch 013] train_loss: 0.30223  val_loss: 0.61370
[TCN] [Epoch 014] train_loss: 0.29945  val_loss: 0.58765
[TCN] [Epoch 015] train_loss: 0.29760  val_loss: 0.56388
[TCN] [Epoch 016] train_loss: 0.29452  val_loss: 0.54242
[TCN] [Epoch 017] train_lo